In [ ]:
%cd /app

In [ ]:
import argparse
import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import torch
torch.multiprocessing.set_start_method('spawn')

import jax
from lob.encoding import Vocab, Message_Tokenizer

from lob import inference_no_errcorr as inference
from lob.init_train import init_train_state, load_checkpoint, load_metadata, load_args_from_checkpoint

from lob import inference_no_errcorr as inference
import lob.encoding as encoding
import preproc as preproc

import jax.numpy as jnp
import numpy as np

from pathlib import Path
import os

import pandas as pd

import pandas as pd
import plotly.graph_objs as go
import yaml
import pickle

from filtration_utils import summary_table, build_zero_padded_series, plot_midprice_series_with_insertions, prepare_volatility_filtered_series, plot_midprice_series_with_mean_std
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from typing import Callable, Tuple, Optional, List, Dict

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# ===== unfiltered =====
buy_exp_name = 'exp_151_20251006_133342_hist_buy_75_b0_b7'
sell_exp_name = 'exp_152_20251006_143539_hist_sell_75_b0_b7'

merged_buy_indx = pd.read_csv(f'/app/data_saved/{buy_exp_name}/samples_after_book_filtration.csv')
merged_sell_indx = pd.read_csv(f'/app/data_saved/{sell_exp_name}/samples_after_book_filtration.csv')
sample_day_map = pd.read_csv(f'/app/batches_equal_sample_day_map.csv')

# Find samples that are in both buy and sell datasets
print("Buy dataset shape:", merged_buy_indx.shape)
print("Sell dataset shape:", merged_sell_indx.shape)

# The datasets contain sample indices after book filtration
# Extract the sample indices from both datasets
buy_sample_indices = set(merged_buy_indx['samples after book filtration'])
sell_sample_indices = set(merged_sell_indx['samples after book filtration'])

print(f"\nNumber of samples in buy dataset: {len(buy_sample_indices)}")
print(f"Number of samples in sell dataset: {len(sell_sample_indices)}")

# Find common sample indices
common_sample_indices_set = buy_sample_indices & sell_sample_indices
print(f"Number of common samples: {len(common_sample_indices_set)}")

# Create a DataFrame with the same columns as merged_buy_indx, filtered to common samples
common_sample_indices = merged_buy_indx[merged_buy_indx['samples after book filtration'].isin(common_sample_indices_set)].copy()
print(f"Common samples DataFrame shape: {common_sample_indices.shape}")


In [ ]:
# ===== unfiltered =====
with open(f'/app/data_saved/{buy_exp_name}/filtered_merged.pkl', 'rb') as f:
    merged_buy_full = pickle.load(f)

with open(f'/app/data_saved/{sell_exp_name}/filtered_merged.pkl', 'rb') as f:
    merged_sell_full = pickle.load(f)

print(f"Buy dataset shape before filtration: {merged_buy_full.shape}")
print(f"Sell dataset shape before filtration: {merged_sell_full.shape}")

# Filter datasets to only include common sample indices
merged_buy = merged_buy_full[merged_buy_full.id.isin(common_sample_indices_set)].copy()
merged_sell = merged_sell_full[merged_sell_full.id.isin(common_sample_indices_set)].copy()

print(f"Filtered buy dataset shape: {merged_buy.shape}")
print(f"Filtered sell dataset shape: {merged_sell.shape}")

# # # ===== specific filtered =====
# with open('/app/data_saved/exp_buy_85_sell_89/merged_buy.pkl', 'rb') as f:
#     merged_buy = pickle.load(f)

# with open('/app/data_saved/exp_buy_85_sell_89/merged_sell.pkl', 'rb') as f:
#     merged_sell = pickle.load(f)

In [ ]:
merged_buy

In [ ]:
# Create a combined dataframe with buy and sell data
# Add prefix to distinguish buy vs sell orders
merged_buy_prefixed = merged_buy.copy()
merged_buy_prefixed['id'] = merged_buy_prefixed['id'].astype(str) + '_buy'

merged_sell_prefixed = merged_sell.copy()
merged_sell_prefixed['id'] = merged_sell_prefixed['id'].astype(str) + '_sell'

# Combine both dataframes
combined_df = pd.concat([merged_buy_prefixed, merged_sell_prefixed], ignore_index=True)

print(f"Combined dataframe shape: {combined_df.shape}")
print(f"Original buy dataset shape: {merged_buy.shape}")
print(f"Original sell dataset shape: {merged_sell.shape}")
print(f"Total rows in combined dataframe: {len(combined_df)}")

# Display sample of the combined dataframe
print("\nSample of combined dataframe:")
print(combined_df.head(10))

print(f"\nCombined dataframe created:")
print(f"- Contains {len(merged_buy_prefixed)} buy orders (with '_buy' suffix)")
print(f"- Contains {len(merged_sell_prefixed)} sell orders (with '_sell' suffix)")
print(f"- Total records: {len(combined_df):,}")

# Create plotly lineplot with 3 lines: avg on all, _buy, _sell ids
import plotly.graph_objects as go
import numpy as np

# Extract merged_data arrays and compute returns (subtract price[0])
all_returns = []
buy_returns = []
sell_returns = []

# Process all data
for _, row in combined_df.iterrows():
    data_array = np.array(row['merged_data'])
    # Convert to returns (subtract initial price)
    returns = data_array - data_array[0]
    
    if row['id'].endswith('_buy'):
        buy_returns.append(returns)
        all_returns.append(returns)
    elif row['id'].endswith('_sell'):
        # Multiply sell returns by -1
        sell_returns_inverted = returns * -1.0
        sell_returns.append(sell_returns_inverted)
        all_returns.append(sell_returns_inverted)

# Convert to numpy arrays and compute averages and standard deviations
all_returns = np.array(all_returns)
buy_returns = np.array(buy_returns)
sell_returns = np.array(sell_returns)

# Compute averages and standard deviations across samples
avg_all_returns = np.mean(all_returns, axis=0)
std_all_returns = np.std(all_returns, axis=0)

avg_buy_returns = np.mean(buy_returns, axis=0)
std_buy_returns = np.std(buy_returns, axis=0)

avg_sell_returns = np.mean(sell_returns, axis=0)
std_sell_returns = np.std(sell_returns, axis=0)

# Create x-axis (time steps)
x_axis = list(range(len(avg_all_returns)))

# Create plotly figure
fig = go.Figure()

# Add lines with error bands (mean ± std)
# All data
fig.add_trace(go.Scatter(
    x=x_axis + x_axis[::-1],
    y=np.concatenate([avg_all_returns + std_all_returns, (avg_all_returns - std_all_returns)[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 0, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=x_axis,
    y=avg_all_returns,
    mode='lines',
    name=f'All (Buy + Sell*-1) - {len(all_returns)} points',
    line=dict(color='blue', width=2)
))

# Buy data
fig.add_trace(go.Scatter(
    x=x_axis + x_axis[::-1],
    y=np.concatenate([avg_buy_returns + std_buy_returns, (avg_buy_returns - std_buy_returns)[::-1]]),
    fill='toself',
    fillcolor='rgba(0, 255, 0, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=x_axis,
    y=avg_buy_returns,
    mode='lines',
    name=f'Buy Orders - {len(buy_returns)} points',
    line=dict(color='green', width=2)
))

# Sell data
fig.add_trace(go.Scatter(
    x=x_axis + x_axis[::-1],
    y=np.concatenate([avg_sell_returns + std_sell_returns, (avg_sell_returns - std_sell_returns)[::-1]]),
    fill='toself',
    fillcolor='rgba(255, 0, 0, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=x_axis,
    y=avg_sell_returns,
    mode='lines',
    name=f'Sell Orders (*-1) - {len(sell_returns)} points',
    line=dict(color='red', width=2)
))

# Update layout
fig.update_layout(
    title='Average Returns Evolution: All vs Buy vs Sell Orders (Sell*-1) with ±1 Std',
    xaxis_title='Time Steps',
    yaxis_title='Average Returns (Price - Initial Price)',
    width=1000,
    height=600,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

print(f"\nPlot created with returns (price - initial price) and standard deviation bands:")
print(f"- All data average (blue): {len(all_returns)} samples")
print(f"- Buy data average (green): {len(buy_returns)} samples")
print(f"- Sell data average (red, *-1): {len(sell_returns)} samples")
print(f"- Note: Sell returns multiplied by -1 for comparison")
print(f"- Each line shows number of points used in calculation in legend")
print(f"- Shaded areas represent ±1 standard deviation")


In [ ]:
all_series = all_returns

# other plots

In [ ]:
# ===== unfiltered =====
with open(f'/app/data_saved/{buy_exp_name}/filtered_merged.pkl', 'rb') as f:
    merged_buy_full = pickle.load(f)

with open(f'/app/data_saved/{sell_exp_name}/filtered_merged.pkl', 'rb') as f:
    merged_sell_full = pickle.load(f)

# Load b_dict files
with open(f'/app/data_saved/{buy_exp_name}/filtered_b_dict.pkl', 'rb') as f:
    b_dict_buy_full = pickle.load(f)

with open(f'/app/data_saved/{sell_exp_name}/filtered_b_dict.pkl', 'rb') as f:
    b_dict_sell_full = pickle.load(f)

# Load m_dict files
with open(f'/app/data_saved/{buy_exp_name}/filtered_m_dict.pkl', 'rb') as f:
    m_dict_buy_full = pickle.load(f)

with open(f'/app/data_saved/{sell_exp_name}/filtered_m_dict.pkl', 'rb') as f:
    m_dict_sell_full = pickle.load(f)

print(f"Buy dataset shape before filtration: {merged_buy_full.shape}")
print(f"Sell dataset shape before filtration: {merged_sell_full.shape}")
print(f"b_dict_buy keys before filtration: {len(b_dict_buy_full)}")
print(f"b_dict_sell keys before filtration: {len(b_dict_sell_full)}")
print(f"m_dict_buy keys before filtration: {len(m_dict_buy_full)}")
print(f"m_dict_sell keys before filtration: {len(m_dict_sell_full)}")

# Filter datasets to only include common sample indices
merged_buy = merged_buy_full[merged_buy_full.id.isin(common_sample_indices_set)].copy()
merged_sell = merged_sell_full[merged_sell_full.id.isin(common_sample_indices_set)].copy()

# Filter b_dict and m_dict to only include common sample indices
b_dict_buy = {k: v for k, v in b_dict_buy_full.items() if k in common_sample_indices_set}
b_dict_sell = {k: v for k, v in b_dict_sell_full.items() if k in common_sample_indices_set}
m_dict_buy = {k: v for k, v in m_dict_buy_full.items() if k in common_sample_indices_set}
m_dict_sell = {k: v for k, v in m_dict_sell_full.items() if k in common_sample_indices_set}

print(f"Filtered buy dataset shape: {merged_buy.shape}")
print(f"Filtered sell dataset shape: {merged_sell.shape}")
print(f"Filtered b_dict_buy keys: {len(b_dict_buy)}")
print(f"Filtered b_dict_sell keys: {len(b_dict_sell)}")
print(f"Filtered m_dict_buy keys: {len(m_dict_buy)}")
print(f"Filtered m_dict_sell keys: {len(m_dict_sell)}")

In [ ]:
# # ===== specific filtered =====

# # Load b_dict_buy.pkl
# with open('/app/data_saved/exp_buy_85_sell_89/b_dict_buy.pkl', 'rb') as f:
#     b_dict_buy = pickle.load(f)

# # Load b_dict_sell.pkl
# with open('/app/data_saved/exp_buy_85_sell_89/b_dict_sell.pkl', 'rb') as f:
#     b_dict_sell = pickle.load(f)

# # Load m_dict_buy.pkl
# with open('/app/data_saved/exp_buy_85_sell_89/m_dict_buy.pkl', 'rb') as f:
#     m_dict_buy = pickle.load(f)

# # Load m_dict_sell.pkl
# with open('/app/data_saved/exp_buy_85_sell_89/m_dict_sell.pkl', 'rb') as f:
#     m_dict_sell = pickle.load(f)


In [ ]:
# Merge m_dicts and b_dicts with _buy and _sell suffixes
m_dict_combined = {}
b_dict_combined = {}

# Add buy dictionaries with _buy suffix
for key, value in m_dict_buy.items():
    m_dict_combined[key] = value

for key, value in b_dict_buy.items():
    b_dict_combined[key] = value

# Add sell dictionaries with _sell suffix (add 1000000 to keys)
for key, value in m_dict_sell.items():
    m_dict_combined[key + 1000000] = value

for key, value in b_dict_sell.items():
    b_dict_combined[key + 1000000] = value

print(f"Combined m_dict keys: {len(m_dict_combined)}")
print(f"Combined b_dict keys: {len(b_dict_combined)}")


In [ ]:
# Load sample_day_map and create combined version for buy and sell orders
# sample_day_map = pd.read_csv('/app/sample_day_map_1024.csv')

# Create a copy for sell orders with sample_id + 1000000
sample_day_map_sell = sample_day_map.copy()
sample_day_map_sell['sample_id'] = sample_day_map_sell['sample_id'] + 1000000

# Combine buy and sell mappings
sample_day_map_combined = pd.concat([sample_day_map, sample_day_map_sell], ignore_index=True)

print(f"Original sample_day_map rows: {len(sample_day_map)}")
print(f"Combined sample_day_map rows: {len(sample_day_map_combined)}")
print(f"Buy sample_ids range: {sample_day_map['sample_id'].min()} - {sample_day_map['sample_id'].max()}")
print(f"Sell sample_ids range: {sample_day_map_sell['sample_id'].min()} - {sample_day_map_sell['sample_id'].max()}")

sample_day_map = sample_day_map_combined

In [ ]:
# Extract volumes for buy and sell orders
buy_volumes = []
sell_volumes = []

# Optional filtration parameter
filter_small_volumes = True  # Set to False to include all volumes
volume_threshold = 1000  # Only used if filter_small_volumes is True

for sample_id, messages in m_dict_combined.items():
    volumes = messages[:, 5]  # 5th index is volume
    
    # Apply optional filtration
    if filter_small_volumes:
        volumes = volumes[volumes <= volume_threshold]
    
    if sample_id < 1000000:  # Buy orders
        buy_volumes.extend(volumes)
    else:  # Sell orders (sample_id >= 1000000)
        sell_volumes.extend(volumes)

# Create histograms
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# Add buy volumes histogram
fig.add_trace(go.Histogram(
    x=buy_volumes,
    nbinsx=100,
    opacity=0.7,
    name='Buy Orders (sample < 1000000)',
    marker_color='blue'
))

# Add sell volumes histogram
fig.add_trace(go.Histogram(
    x=sell_volumes,
    nbinsx=100,
    opacity=0.7,
    name='Sell Orders (sample >= 1000000)',
    marker_color='red'
))

# Update layout
filter_text = f" (filtered >= {volume_threshold})" if filter_small_volumes else ""
fig.update_layout(
    title=f'Volume Distribution: Buy vs Sell Orders{filter_text}',
    xaxis_title='Volume',
    yaxis_title='Frequency',
    width=1200,
    height=600,
    barmode='overlay',
    showlegend=True
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

fig.show()

print(f"Buy volumes count: {len(buy_volumes)}")
print(f"Sell volumes count: {len(sell_volumes)}")
if filter_small_volumes:
    print(f"Filtration applied: volumes >= {volume_threshold}")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- helpers ---------------------------------------------------------------

def split_l2_vector(vec: np.ndarray,
                    ask_before_bid: bool = True,
                    odd_policy: str = "drop_center"):
    """
    Split a 1D L2 vector into (ask_vec, bid_vec).

    - If len(vec) is even: split into two equal halves.
    - If len(vec) is odd and odd_policy == "drop_center":
        drop the center column (index len//2) and split the remainder evenly.
    - odd_policy can be "drop_center", "drop_last", or "error".
    Returns (ask_vec, bid_vec, dropped_index_or_None)
    """
    n = int(vec.shape[0])

    if n % 2 == 0:
        k = n // 2
        if ask_before_bid:
            ask, bid = vec[:k], vec[k:]
        else:
            bid, ask = vec[:k], vec[k:]
        return ask, bid, None

    # odd length:
    if odd_policy == "drop_center":
        mid = n // 2
        # build a new vector without the center element
        if ask_before_bid:
            ask = vec[:mid]
            bid = vec[mid+1:]
        else:
            bid = vec[:mid]
            ask = vec[mid+1:]
        return ask, bid, mid

    elif odd_policy == "drop_last":
        vec2 = vec[:-1]
        return split_l2_vector(vec2, ask_before_bid=ask_before_bid, odd_policy="drop_center")

    elif odd_policy == "error":
        raise ValueError(f"Odd L2 width ({n}); cannot split evenly without a policy.")

    else:
        raise ValueError(f"Unknown odd_policy={odd_policy!r}")


def compute_queued_volumes_over_time(book_array: np.ndarray,
                                     l2_start: int = 240,
                                     l2_end: int = 263,
                                     ask_before_bid: bool = True,
                                     odd_policy: str = "drop_center",
                                     use_abs: bool = True):
    """
    For each time t, sum queued volume on ASK and BID from the L2 slice [l2_start:l2_end).
    Handles odd width by the given odd_policy (default: drop the center column).
    """
    T = book_array.shape[0]
    bid_vol = np.empty(T, dtype=float)
    ask_vol = np.empty(T, dtype=float)

    for t in range(T):
        vec = book_array[t, l2_start:l2_end].astype(float)
        ask_vec, bid_vec, _ = split_l2_vector(vec, ask_before_bid=ask_before_bid, odd_policy=odd_policy)
        if use_abs:
            bid_vol[t] = float(np.sum(np.abs(bid_vec)))
            ask_vol[t] = float(np.sum(np.abs(ask_vec)))
        else:
            bid_vol[t] = float(np.sum(bid_vec))
            ask_vol[t] = float(np.sum(ask_vec))

    total = bid_vol + ask_vol
    return bid_vol, ask_vol, total


# --- main widget -----------------------------------------------------------

def interactive_lob_plot(b_seq_inp, msg_seq_raw,
                         l2_start: int = 240, l2_end: int = 263,
                         ask_before_bid: bool = True,
                         odd_policy: str = "drop_center",
                         use_abs: bool = True):
    """
    Extends your original function:
    - keeps the two book-state panels
    - computes queued volumes (Bid/Ask/Total) over time from the same L2 slice
    - shows a live volume chart and prints volumes at current t
    """

    # allow DataFrame or dict
    if isinstance(b_seq_inp, pd.DataFrame):
        b_seq_inp = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    if isinstance(msg_seq_raw, pd.DataFrame):
        msg_seq_raw = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}

    # controls
    id_dd       = widgets.Dropdown(options=sorted(b_seq_inp.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev    = widgets.Button(description="←", layout=widgets.Layout(width="50px"))
    btn_next    = widgets.Button(description="→", layout=widgets.Layout(width="50px"))
    msg_box     = widgets.HTML()
    vol_box     = widgets.HTML()  # will show numeric volumes at t

    # --- Figure A: two book panels
    figA = make_subplots(rows=1, cols=2, subplot_titles=["Book state t–1", "Book state t"])
    figA.add_trace(go.Bar(x=[], y=[]), row=1, col=1)
    figA.add_trace(go.Bar(x=[], y=[]), row=1, col=2)
    figA.update_layout(width=900, height=380, showlegend=False, template='plotly_white', margin=dict(l=30,r=30,t=50,b=30))
    figA_widget = go.FigureWidget(figA)

    # --- Figure B: volumes over time
    figB = go.FigureWidget(
        go.Figure(layout=go.Layout(
            width=900, height=280, template='plotly_white', margin=dict(l=30,r=30,t=30,b=30)
        ))
    )
    # add empty traces for bid/ask/total
    figB.add_scatter(x=[], y=[], mode='lines', name='Ask queued volume')
    figB.add_scatter(x=[], y=[], mode='lines', name='Bid queued volume')
    figB.add_scatter(x=[], y=[], mode='lines', name='Total queued volume')
    # vertical cursor at t
    figB.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x", yref="paper",
                   line=dict(color="gray", width=1, dash="dash"))

    # internal cache per sample_id to avoid recomputing every slider move
    _vol_cache = {}

    def update_slider_range(*_):
        arr = b_seq_inp[id_dd.value]
        time_slider.min = 1
        time_slider.max = arr.shape[0] - 1
        time_slider.value = 1

        # (re)compute volumes for this sample once
        if id_dd.value not in _vol_cache:
            bid_v, ask_v, tot_v = compute_queued_volumes_over_time(
                arr, l2_start=l2_start, l2_end=l2_end,
                ask_before_bid=ask_before_bid, odd_policy=odd_policy, use_abs=use_abs
            )
            _vol_cache[id_dd.value] = (bid_v, ask_v, tot_v)

        # refresh the volume chart lines for the new sample
        bid_v, ask_v, tot_v = _vol_cache[id_dd.value]
        T = len(bid_v)
        xs = np.arange(T)

        with figB.batch_update():
            figB.data[0].x = xs; figB.data[0].y = bid_v
            figB.data[1].x = xs; figB.data[1].y = ask_v
            figB.data[2].x = xs; figB.data[2].y = tot_v
            # reset vertical cursor to current t
            figB.layout.shapes[0].x0 = time_slider.value
            figB.layout.shapes[0].x1 = time_slider.value
            figB.update_xaxes(title="Time")
            figB.update_yaxes(title="Queued volume (sum over L2)")

    def update_plot(*_):
        sid = id_dd.value
        t   = time_slider.value
        arr = b_seq_inp[sid]
        msgs= msg_seq_raw[sid]

        # slice the same L2 window you visualize
        # NOTE: this window is odd (263-240=23). We handle it with odd_policy="drop_center" by default.
        s0 = arr[t-1, l2_start:l2_end]
        s1 = arr[t,   l2_start:l2_end]

        # difference coloring as you had
        diff = np.abs(s1) - np.abs(s0)
        x = np.arange(len(s0)) - len(s0)//2
        colors = ['orange' if abs(d)<1e-8 else ('red' if d>0 else 'blue') for d in diff]

        # book panels
        with figA_widget.batch_update():
            figA_widget.data = []  # clear
            figA_widget.add_bar(x=x, y=s0, row=1, col=1, marker_color='orange')
            figA_widget.add_bar(x=x, y=s1, row=1, col=2, marker_color=colors)
            figA_widget.layout.annotations[0].text = f"Book state {t-1}"
            figA_widget.layout.annotations[1].text = f"Book state {t}"

        # message info
        m = msgs[t].astype(int)
        # fields: [0]=timestamp, [1]=etype, [2]=dir, [3]=abspr, [4]=relpr, [5]=size, …
        et, dr, abspr, relpr, sz = m[1], m[2], m[3], m[4], m[5]
        et_map = {1:"Limit", 2:"PartialCancel", 3:"Delete", 4:"Execution"}
        dr_map = {1:"Buy", 0:"Sell"}
        info = (
            f"{et_map.get(et,'?')} • {dr_map.get(dr,'?')} • abs={abspr} • rel={relpr} • size={sz}"
        )
        msg_box.value = f"<b>{info}</b><br>raw: {m.tolist()}"

        # volumes at t (from cache)
        bid_v, ask_v, tot_v = _vol_cache[sid]
        bv, av, tv = bid_v[t], ask_v[t], tot_v[t]
        vol_box.value = f"<b>Queued volume @ t={t}:</b>  Bid={bv:.0f} • Ask={av:.0f} • Total={tv:.0f}"

        # move the vertical cursor on the volume chart
        with figB.batch_update():
            figB.layout.shapes[0].x0 = t
            figB.layout.shapes[0].x1 = t

    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1

    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1

    # wire up events
    id_dd.observe(lambda _: update_slider_range(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    # initial draw
    update_slider_range()
    update_plot()

    # layout
    controls = widgets.HBox([id_dd, btn_prev, btn_next, time_slider])
    display(controls)
    display(figA_widget)
    display(figB)
    display(vol_box, msg_box)

In [ ]:
interactive_lob_plot(b_dict_buy, m_dict_buy,
                     l2_start=210, l2_end=293,     
                     ask_before_bid=True,          
                     odd_policy="drop_center",     
                     use_abs=True)                 

In [ ]:
interactive_lob_plot(b_dict_sell, m_dict_sell,
                     l2_start=210, l2_end=293,     
                     ask_before_bid=True,          
                     odd_policy="drop_center",     
                     use_abs=True)    

In [ ]:
interactive_lob_plot(b_dict_combined, m_dict_combined,
                     l2_start=210, l2_end=293,     
                     ask_before_bid=True,          
                     odd_policy="drop_center",     
                     use_abs=True)    

# Market impact graph

In [ ]:
# def calculate_impact(messages, valid_insertions, reference_price):
#     """
#     Calculate market impact for each insertion.
    
#     Parameters
#     ----------
#     messages : np.array
#         Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
#     valid_insertions : list
#         List of insertion indices
#     reference_price : float
#         Reference price at first insertion
        
#     Returns
#     -------
#     impact : np.array
#         Absolute impact for each insertion
#     vwap_series : np.array
#         VWAP series for each insertion
#     Q_cum : np.array
#         Cumulative quantity for each insertion
#     log_imp : np.array
#         Log of impact values for plotting
#     """
#     PRICE_COL = 3
#     SIZE_COL = 5
#     insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
#     insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
#     Q_cum = np.cumsum(insert_sizes)
#     notional = np.cumsum(insert_sizes * insert_prices)

#     vwap_series = notional / np.maximum(Q_cum, 1e-12)
#     # vwap_series = insert_prices
    
#     impact = np.abs(vwap_series - reference_price) / reference_price
    
#     # Calculate log impact for plotting (y-axis)
#     eps = 1e-12
#     log_imp = np.log(np.maximum(impact, eps))
    
#     return impact, vwap_series, Q_cum, log_imp


# def calculate_market_volume(messages, hist_steps, valid_insertions, execution_sum):
#     """
#     Calculate market execution volume V_exp for each insertion and return x-axis values for plotting.
    
#     Parameters
#     ----------
#     messages : np.array
#         Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
#     hist_steps : int
#         Starting index for market volume calculation
#     valid_insertions : list
#         List of insertion indices
#     execution_sum : float
#         Total execution sum for the day
        
#     Returns
#     -------
#     V_exp : np.array
#         Market volume from hist_steps to (idx-1) for each insertion
#     log_qv : np.array
#         Log of Q/V_exp ratio for plotting (x-axis)
#     """
#     EVENT_TYPE_COL = 1
#     SIZE_COL = 5
    
#     evt_types = messages[:, EVENT_TYPE_COL].astype(int)
#     exec_sizes = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
#     cum_exec_vol = np.cumsum(exec_sizes)
    
#     V_exp = np.array([float(cum_exec_vol[idx-1] - cum_exec_vol[hist_steps-1] if (idx-1) >= hist_steps else 0.0)
#                       for idx in valid_insertions])

#     # Use execution_sum instead of fixed value
#     V_exp = np.full_like(V_exp, execution_sum)
    
#     # Calculate cumulative quantity for Q/V_exp ratio
#     insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
#     Q_cum = np.cumsum(insert_sizes)
    
#     # Calculate log(Q/V_exp) for plotting (x-axis)
#     eps = 1e-12
#     rel_size = Q_cum / np.maximum(V_exp, eps)
#     log_qv = np.log(np.maximum(rel_size, eps))

#     return V_exp, log_qv

In [ ]:
# # Different methods of impact regression
# import numpy as np

# # ---------- small utils ----------
# def _as_float(a):
#     return np.asarray(a, dtype=float)

# def _mask_xy(x, y):
#     x = _as_float(x); y = _as_float(y)
#     m = np.isfinite(x) & np.isfinite(y) & (x != 0.0)
#     return x[m], y[m], m

# def _beta_wls_fixed_internal(x, y_adj, w):
#     # Weighted regression of y_adj on x with intercept fixed at 0
#     x = _as_float(x); y_adj = _as_float(y_adj); w = _as_float(w)
#     m = np.isfinite(x) & np.isfinite(y_adj) & np.isfinite(w) & (x != 0) & (w > 0)
#     if m.sum() < 2: return np.nan
#     xw = x[m] * np.sqrt(w[m]); yw = y_adj[m] * np.sqrt(w[m])
#     denom = np.dot(xw, xw)
#     if denom <= 0: return np.nan
#     return float(np.dot(xw, yw) / denom)

# # ---------- estimators (fixed intercept y = alpha + beta * x) ----------
# def _beta_ols_fixed(x, y, alpha):
#     x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
#     if x.size < 2: return np.nan
#     denom = np.dot(x, x)
#     if denom <= 0: return np.nan
#     return float(np.dot(x, y_adj) / denom)

# def _beta_huber_fixed(x, y, alpha, c=1.345, max_iter=50, tol=1e-8):
#     # Huber M via IRLS (defaults chosen for ~95% Gaussian efficiency)
#     x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
#     if x.size < 2: return np.nan
#     beta = _beta_ols_fixed(x, y, alpha)
#     if not np.isfinite(beta): beta = 0.0
#     for _ in range(max_iter):
#         r = y_adj - beta * x
#         med = np.median(r)
#         sigma = 1.4826 * np.median(np.abs(r - med)) or (np.std(r) + 1e-12)
#         u = r / (sigma + 1e-12)
#         w = np.ones_like(u)
#         big = np.abs(u) > c
#         w[big] = (c / (np.abs(u[big]) + 1e-12))
#         beta_new = _beta_wls_fixed_internal(x, y_adj, w)
#         if not np.isfinite(beta_new): break
#         if abs(beta_new - beta) <= tol * (1.0 + abs(beta)):
#             beta = beta_new; break
#         beta = beta_new
#     return float(beta)

# def _beta_lad_fixed(x, y, alpha, iters=100, eps=1e-8):
#     # LAD (L1) via IRLS: w_i = 1/max(|r_i|, eps)
#     x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
#     if x.size < 2: return np.nan
#     beta = _beta_ols_fixed(x, y, alpha)
#     if not np.isfinite(beta): beta = 0.0
#     for _ in range(iters):
#         r = y_adj - beta * x
#         w = 1.0 / np.maximum(np.abs(r), eps)
#         beta_new = _beta_wls_fixed_internal(x, y_adj, w)
#         if not np.isfinite(beta_new): break
#         if abs(beta_new - beta) <= 1e-7 * (1.0 + abs(beta)):
#             beta = beta_new; break
#         beta = beta_new
#     return float(beta)

# def _beta_ratio_median(x, y, alpha, x_floor=1e-6):
#     # Median of ratios with |x| floor (avoid blow-ups near 0)
#     x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
#     m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
#     r = y_adj[m] / x[m]
#     if r.size == 0: return np.nan
#     return float(np.median(np.sort(r)))

# def _beta_ratio_trim(x, y, alpha, x_floor=1e-6, trim=0.10):
#     # Trimmed-mean of ratios (default 10% each tail) with |x| floor
#     x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
#     m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
#     r = np.sort(y_adj[m] / x[m])
#     if r.size == 0: return np.nan
#     k = int(trim * r.size)
#     r = r[k: r.size - k] if r.size - 2*k > 0 else r
#     return float(np.mean(r))

# def _beta_deming_fixed(x, y, alpha, lambda_yx=1.0):
#     # Orthogonal regression with fixed intercept (through origin on y_adj)
#     x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
#     if x.size < 2: return np.nan
#     s_xx = np.dot(x, x) / x.size
#     s_yy = np.dot(y_adj, y_adj) / x.size
#     s_xy = np.dot(x, y_adj) / x.size
#     if s_xy == 0.0: return np.nan
#     A = s_yy - lambda_yx * s_xx
#     B = 2.0 * s_xy
#     disc = A*A + (B*B) * lambda_yx
#     beta = (A + np.sqrt(disc)) / B
#     return float(beta)

# # ---------- single public entry ----------
# def beta_fit(x, y, alpha, method="ols"):
#     """
#     Estimate beta in y = alpha + beta * x with a fixed intercept.

#     Parameters
#     ----------
#     x, y : array-like
#     alpha : float or array-like (broadcastable)
#     method : {'ols','huber','lad','ratio-median','ratio-trim','deming'}

#     Returns
#     -------
#     beta : float
#     """
#     m = method.lower()
#     if m == "ols":
#         return _beta_ols_fixed(x, y, alpha)
#     elif m == "huber":
#         return _beta_huber_fixed(x, y, alpha)           # c=1.345, 50 iters, tol=1e-8
#     elif m == "lad":
#         return _beta_lad_fixed(x, y, alpha)             # 100 iters, eps=1e-8
#     elif m == "ratio-median":
#         return _beta_ratio_median(x, y, alpha)          # x_floor=1e-6
#     elif m == "ratio-trim":
#         return _beta_ratio_trim(x, y, alpha)            # trim=10%, x_floor=1e-6
#     elif m == "deming":
#         return _beta_deming_fixed(x, y, alpha)          # lambda_yx=1.0
#     else:
#         raise ValueError(f"Unknown method '{method}'. Use one of: "
#                          "ols, huber, lad, ratio-median, ratio-trim, deming.")

In [ ]:
# def market_impact_dashboard_from_raw(
#     b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step (ticks)
#     msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
#     all_series,           # unused for mid now
#     x,                    # time axis (len T) - unused here
#     hist_steps=550,
#     gen_block=50,
#     num_insertions=20,
#     *,
#     beta_theory=0.5,
#     samples_used=None,
#     special_first=(79, 15),
#     show_fit=True,
#     tick_size=100,        # convert ticks -> dollars before calling helper funcs
# ):
#     """
#     Same UI/figure as before, but x/y are computed via your helper functions:
#       - impact/log_imp from calculate_impact(...)
#       - V_exp/log_qv from calculate_market_volume(...), now fed with day-level execution_sum from sample_day_map
#     α is fixed to ln(eta_day), where eta_day = ln(H/L)/0.8325546 using H,L from the same sample_day_map.
#     β is the fixed-intercept slope: mean((y - α) / x).
#     """

#     # ------------- normalize inputs -------------
#     if isinstance(b_seq_inp, pd.DataFrame):
#         b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
#     else:
#         b_dict_local = b_seq_inp
#     if isinstance(msg_seq_raw, pd.DataFrame):
#         m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
#     else:
#         m_dict_local = msg_seq_raw

#     EVENT_TYPE_COL = 1
#     PRICE_COL      = 3
#     SIZE_COL       = 5

#     eps = 1e-12
#     tol = 1e-12

#     # -------------------- compute x_df, y_df, coeffs_df via helpers --------------------
#     def compute_tables():
#         sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
#         col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
#         x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
#         y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
#         coeff_rows = []

#         for sid in sample_ids:
#             messages_ticks = m_dict_local[sid]
#             book = b_dict_local[sid]
#             T = len(messages_ticks)

#             # insertion schedule
#             insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
#             valid_insertions = [pos for pos in insertion_positions if pos < T]
#             if not valid_insertions:
#                 coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan,
#                                    "n_used": 0, "n_total": 0})
#                 continue

#             # Reference price at first insertion (ticks -> $)
#             ref_idx = valid_insertions[0]
#             reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

#             # -------- NEW: get day info from sample_day_map --------
#             # Look up this sample_id in sample_day_map
#             try:
#                 day_row = sample_day_map[sample_day_map['sample_id'] == sid]
#                 if not day_row.empty:
#                     H_ticks = float(day_row.iloc[0]['highest_price'])
#                     L_ticks = float(day_row.iloc[0]['lowest_price'])
#                     execution_sum = float(day_row.iloc[0]['execution_sum'])
#                 else:
#                     # Fallback if not found: infer H/L from pre-gen window and use window exec sum
#                     H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
#                     L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
#                     exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
#                     execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
#             except Exception as _e:
#                 # Fallback if not found: infer H/L from pre-gen window and use window exec sum
#                 H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
#                 L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
#                 exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
#                 execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

#             # Convert to dollars for Parkinson eta
#             H = float(H_ticks) / tick_size
#             L = float(L_ticks) / tick_size
#             if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
#                 eta_day = np.log(H / L) / 0.8325546
#                 alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
#             else:
#                 eta_day = eps
#                 alpha_fixed = float(np.log(eta_day))

#             # Convert messages to dollars for the helper functions
#             messages_dollars = messages_ticks.astype(float).copy()
#             messages_dollars[:, PRICE_COL] /= tick_size

#             # --- use your helper functions (unchanged graphs) ---
#             impact, vwap_series, Q_cum, log_imp = calculate_impact(
#                 messages_dollars, valid_insertions, reference_price
#             )
#             # -------- NEW: pass execution_sum from sample_day_map into V_exp calc --------
#             V_exp, log_qv = calculate_market_volume(
#                 messages_dollars, hist_steps, valid_insertions, execution_sum
#             )

#             # fill tables per insertion (same)
#             mask_zero = impact <= tol
#             mask_pos  = ~mask_zero
#             for j, _idx in enumerate(valid_insertions):
#                 col = f"ins_{j+1}"
#                 if mask_zero[j] or not np.isfinite(log_qv[j]) or not np.isfinite(log_imp[j]):
#                     x_df.loc[sid, col] = "ZERO"
#                     y_df.loc[sid, col] = "ZERO"
#                 else:
#                     x_df.loc[sid, col] = float(log_qv[j])
#                     y_df.loc[sid, col] = float(log_imp[j])

#             # per-sample β with fixed intercept (same)
#             used_x = log_qv[mask_pos]
#             used_y = log_imp[mask_pos]
#             n_used = int(used_x.size)
#             n_total = int(len(valid_insertions))
#             if n_used >= 2 and np.all(np.isfinite(used_x)) and np.all(np.isfinite(used_y)):
#                 valid_mask = (used_x != 0) & np.isfinite(used_x) & np.isfinite(used_y)
#                 if np.sum(valid_mask) >= 2:
#                     # beta_hat = float(np.mean((used_y[valid_mask] - alpha_fixed) / used_x[valid_mask]))
#                     beta_hat = beta_fit(used_x[valid_mask], used_y[valid_mask], alpha_fixed, method=est_method)                    
#                 else:
#                     beta_hat = np.nan
#             else:
#                 beta_hat = np.nan

#             coeff_rows.append({
#                 "sample_id": sid,
#                 "alpha_hat": alpha_fixed,   # fixed ln(η_day) from sample_day_map
#                 "beta_hat": beta_hat,
#                 "n_used": n_used,
#                 "n_total": n_total,
#             })

#         coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
#         return x_df, y_df, coeffs_df

#     x_df, y_df, coeffs_df = compute_tables()

#     # -------------------- tidy points (unchanged) --------------------
#     all_ids = list(x_df.index)
#     ordered_ids = [sid for sid in special_first if sid in all_ids]
#     ordered_ids += [sid for sid in sorted(all_ids) if sid not in ordered_ids]
#     if samples_used is not None:
#         ordered_ids = ordered_ids[:samples_used]

#     rows = []
#     for sid in ordered_ids:
#         for j, col in enumerate(x_df.columns, start=1):
#             xv = x_df.loc[sid, col]
#             yv = y_df.loc[sid, col]
#             if isinstance(xv, (int, float, np.floating)) and isinstance(yv, (int, float, np.floating)):
#                 if np.isfinite(xv) and np.isfinite(yv):
#                     rows.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
#     points_df = pd.DataFrame(rows)
#     if points_df.empty:
#         print("No numeric points to plot.")
#         return None, pd.DataFrame(), pd.DataFrame(), {}

#     data_max_ins = int(points_df["insertion"].max())
#     max_insertions = int(min(num_insertions, data_max_ins))
#     a_values = np.arange(1, max_insertions + 1)

#     # -------------------- histogram data (unchanged) --------------------
#     betas_clean = coeffs_df["beta_hat"].replace([np.inf, -np.inf], np.nan).dropna().astype(float)
#     beta_mean = float(betas_clean.mean()) if not betas_clean.empty else np.nan

#     # -------------------- figure (unchanged) --------------------
#     fig = go.FigureWidget(make_subplots(
#         rows=2, cols=2,
#         specs=[[{"type": "xy"}, {"type": "xy"}],
#                [None,          {"type": "xy"}]],
#         column_widths=[0.68, 0.32],
#         row_heights=[0.55, 0.45],
#         horizontal_spacing=0.07,
#         vertical_spacing=0.12,
#         subplot_titles=("Scatter & Global Fit", "β(a) evolution", "Distribution of β̂ across samples")
#     ))

#     LEGEND_MAX = 15
#     palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b",
#                "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#ff7f0e"]
#     GREY = "rgba(0,0,0,0.25)"

#     trace_meta = []
#     for i, sid in enumerate(ordered_ids):
#         sub = points_df[points_df["sample_id"] == sid].sort_values("insertion")
#         x_vals = sub["x"].to_numpy()
#         y_vals = sub["y"].to_numpy()
#         ins = sub["insertion"].to_numpy()
#         base_color = palette[i % len(palette)] if i != 1 else "#d62728"
#         tr = go.Scatter(
#             x=x_vals, y=y_vals, mode="markers",
#             name=f"sample {sid} ({len(sub)}/{len(sub)} pts)",
#             legendgroup=str(sid), showlegend=(i < LEGEND_MAX),
#             marker=dict(size=7, color=base_color),
#             hovertemplate=(
#                 "sample=%{customdata[0]}<br>"
#                 "ins=%{customdata[1]}<br>"
#                 "log(Q/V)=%{x:.4f}<br>"
#                 "log(Impact)=%{y:.4f}<extra></extra>"
#             ),
#             customdata=np.stack([sub["sample_id"].to_numpy(), ins], axis=1),
#         )
#         fig.add_trace(tr, row=1, col=1)
#         trace_meta.append({"sid": sid, "ins": ins, "x": x_vals, "y": y_vals,
#                            "base_color": base_color, "total": len(sub)})

#     # global-fit trace (same)
#     fit_trace_index = len(fig.data)
#     fig.add_trace(
#         go.Scatter(x=[], y=[], mode="lines", name="Global fit",
#                    line=dict(dash="dash", width=2)),
#         row=1, col=1
#     )

#     # ----- fitting helpers (unchanged, uses α_global = mean ln(η)) -----
#     alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

#     def fit_for_mask(mask):
#         X = points_df.loc[mask, "x"].to_numpy()
#         Y = points_df.loc[mask, "y"].to_numpy()
#         if len(Y) < 2:
#             return np.nan, np.nan, np.nan, 0, np.array([]), np.array([])
#         valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
#         if np.sum(valid_mask) < 2:
#             return alpha_global, np.nan, np.nan, len(Y), np.array([]), np.array([])
#         x_valid = X[valid_mask]; y_valid = Y[valid_mask]


#         # ================================ #

        
#         # beta = float(np.mean((y_valid - alpha_global) / x_valid))
#         beta = beta_fit(x_valid, y_valid, alpha_global, method=est_method)
        

#         # ================================ #
        
#         y_pred = alpha_global + beta * X
#         ss_tot = float(((Y - Y.mean()) ** 2).sum())
#         ss_res = float(((Y - y_pred) ** 2).sum())
#         r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
#         x_min, x_max = X.min(), X.max()
#         x_line = np.linspace(x_min, x_max, 200)
#         y_line = alpha_global + beta * x_line
#         return alpha_global, beta, r2, len(Y), x_line, y_line

#     # precompute β(a) & fit lines (unchanged)
#     a_values = np.arange(1, max_insertions + 1)
#     betas_evo = np.full_like(a_values, np.nan, dtype=float)
#     fit_lines = {}
#     for idx, a in enumerate(a_values):
#         mask = points_df["insertion"] >= a
#         alpha, beta, r2, n, x_line, y_line = fit_for_mask(mask)
#         betas_evo[idx] = beta
#         fit_lines[a] = (x_line, y_line, alpha, beta, r2, n)

#     # β(a) and indicator (unchanged)
#     beta_line_idx = len(fig.data)
#     fig.add_trace(
#         go.Scatter(x=a_values, y=betas_evo, mode="lines+markers", name="β(a)"),
#         row=1, col=2
#     )
#     beta_vline_idx = len(fig.data)
#     y_min = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
#     y_max = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
#     fig.add_trace(
#         go.Scatter(x=[a_values[0], a_values[0]], y=[y_min, y_max],
#                    mode="lines", line=dict(color="green", dash="dot"), name="current a"),
#         row=1, col=2
#     )

#     # histogram β̂ (unchanged)
#     if not betas_clean.empty:
#         fig.add_trace(go.Histogram(x=betas_clean.values, nbinsx=30, name="β̂"), row=2, col=2)
#         fig.add_trace(go.Scatter(x=[beta_mean, beta_mean], y=[0, max(1, len(betas_clean))],
#                                  mode="lines", name=f"Mean β̂ = {beta_mean:.4f}",
#                                  line=dict(dash="dash")), row=2, col=2)
#         fig.add_trace(go.Scatter(x=[beta_theory, beta_theory], y=[0, max(1, len(betas_clean))],
#                                  mode="lines", name=f"β = {beta_theory:.2f} (theoretical)",
#                                  line=dict(dash="dot")), row=2, col=2)

#     # axes & layout (unchanged)
#     fig.update_xaxes(title_text="log(Q / V_exp)", row=1, col=1)
#     fig.update_yaxes(title_text="log(Impact)", row=1, col=1)
#     fig.update_xaxes(title_text="a (insertion threshold)", row=1, col=2)
#     fig.update_yaxes(title_text="β (slope)", row=1, col=2)
#     fig.update_xaxes(title_text="β̂", row=2, col=2)
#     fig.update_yaxes(title_text="Frequency", row=2, col=2)

#     fig.update_layout(template="plotly_white", width=1500, height=800,
#                       margin=dict(t=70, r=50, b=60, l=60),
#                       legend=dict(orientation="v"))

#     # fit box (unchanged)
#     def set_fit_annotation(text_html: str):
#         fig.layout.annotations = tuple(
#             a for a in (fig.layout.annotations or [])
#             if getattr(a, "name", "") != "fit_box"
#         )
#         fig.add_annotation(
#             x=0.02, y=0.48, xref="paper", yref="paper",
#             text=text_html, showarrow=False, align="left",
#             bordercolor="lightgray", borderwidth=1, borderpad=8,
#             bgcolor="rgba(245,245,245,1)", name="fit_box"
#         )

#     def update_left_fit(a_val: int):
#         x_line, y_line, alpha, beta, r2, n = fit_lines.get(a_val, ([], [], np.nan, np.nan, np.nan, 0))
#         fig.data[fit_trace_index].x = x_line
#         fig.data[fit_trace_index].y = y_line
#         if show_fit and n > 0 and np.isfinite(beta):
#             fit_html = (
#                 "<b>Market Impact Fit (Fixed Intercept)</b><br>"
#                 f"<b>a:</b> {a_val}<br>"
#                 f"<b>Model:</b> log(Impact) = <b>{alpha:.6f}</b> + <b>{beta:.6f}</b> · log(Q/V)<br>"
#                 f"<b>α (fixed):</b> {alpha:.6f} (mean ln(η) across samples)<br>"
#                 f"<b>β:</b> {beta:.6f}<br>"
#                 f"<b>R²:</b> {r2:.4f}<br>"
#                 f"<b>Used points:</b> {n}"
#             )
#         else:
#             fit_html = f"<b>Market Impact Fit (Fixed Intercept)</b><br><b>a:</b> {a_val}<br>No active points."
#         set_fit_annotation(fit_html)

#     def recolor_and_refit(a_val: int):
#         for t_idx, meta in enumerate(trace_meta):
#             active_mask = meta["ins"] >= a_val
#             colors = [meta["base_color"] if ok else GREY for ok in active_mask]
#             fig.data[t_idx].marker.color = colors
#             active_count = int(np.count_nonzero(active_mask))
#             fig.data[t_idx].name = f"sample {meta['sid']} ({meta['total']}/{active_count} pts)"
#         if show_fit:
#             update_left_fit(a_val)

#         y_min_local = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
#         y_max_local = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
#         fig.data[beta_vline_idx].x = [a_val, a_val]
#         fig.data[beta_vline_idx].y = [y_min_local, y_max_local]

#     # controls (unchanged)
#     a_slider = widgets.IntSlider(value=1, min=1, max=max_insertions, step=1, description="a")
#     prev_btn = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
#     next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))

#     def on_prev(_):
#         if a_slider.value > a_slider.min:
#             a_slider.value -= 1

#     def on_next(_):
#         if a_slider.value < a_slider.max:
#             a_slider.value += 1

#     def on_a_change(change):
#         if change["name"] == "value":
#             recolor_and_refit(change["new"])

#     prev_btn.on_click(on_prev)
#     next_btn.on_click(on_next)
#     a_slider.observe(on_a_change, names="value")

#     # initial render
#     recolor_and_refit(a_slider.value)
#     display(widgets.HBox([prev_btn, next_btn, a_slider]), fig)

#     controls = {"a_slider": a_slider, "prev_btn": prev_btn, "next_btn": next_btn}
#     return fig, points_df, coeffs_df, controls

In [ ]:
from IPython.display import display
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets


def calculate_impact(messages, valid_insertions, reference_price):
    """
    Calculate market impact for each insertion.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    valid_insertions : list
        List of insertion indices
    reference_price : float
        Reference price at first insertion
        
    Returns
    -------
    impact : np.array
        Absolute impact for each insertion
    vwap_series : np.array
        VWAP series for each insertion
    Q_cum : np.array
        Cumulative quantity for each insertion
    log_imp : np.array
        Log of impact values for plotting
    """
    PRICE_COL = 3
    SIZE_COL = 5
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    notional = np.cumsum(insert_sizes * insert_prices)

    vwap_series = notional / np.maximum(Q_cum, 1e-12)
    # vwap_series = insert_prices
    
    impact = np.abs(vwap_series - reference_price) / reference_price
    
    # Calculate log impact for plotting (y-axis)
    eps = 1e-12
    log_imp = np.log(np.maximum(impact, eps))
    
    return impact, vwap_series, Q_cum, log_imp


def calculate_market_volume(messages, hist_steps, valid_insertions, execution_sum):
    """
    Calculate market execution volume V_exp for each insertion and return x-axis values for plotting.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    hist_steps : int
        Starting index for market volume calculation
    valid_insertions : list
        List of insertion indices
    execution_sum : float
        Total execution sum for the day
        
    Returns
    -------
    V_exp : np.array
        Market volume from hist_steps to (idx-1) for each insertion
    log_qv : np.array
        Log of Q/V_exp ratio for plotting (x-axis)
    """
    EVENT_TYPE_COL = 1
    SIZE_COL = 5
    
    evt_types = messages[:, EVENT_TYPE_COL].astype(int)
    exec_sizes = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
    cum_exec_vol = np.cumsum(exec_sizes)
    
    V_exp = np.array([float(cum_exec_vol[idx-1] - cum_exec_vol[hist_steps-1] if (idx-1) >= hist_steps else 0.0)
                      for idx in valid_insertions])

    # Use execution_sum instead of fixed value
    V_exp = np.full_like(V_exp, execution_sum)
    
    # Calculate cumulative quantity for Q/V_exp ratio
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    
    # Calculate log(Q/V_exp) for plotting (x-axis)
    eps = 1e-12
    rel_size = Q_cum / np.maximum(V_exp, eps)
    log_qv = np.log(np.maximum(rel_size, eps))

    return V_exp, log_qv


def calculate_book_volumes(book_array):
    """
    Calculate total bid and ask volumes from book data.
    
    Parameters
    ----------
    book_array : np.array
        Book array with L2 data
        
    Returns
    -------
    bid_volumes : np.array
        Total bid volume for each time step
    ask_volumes : np.array
        Total ask volume for each time step
    """
    # Assuming book data structure: first half is ask side, second half is bid side
    # Adjust slice according to your actual book layout
    T = book_array.shape[0]
    # l2_slice = slice(20, 800)  # adjust to your layout
    l2_slice = slice(0, 1000)  # adjust to your layout
    book_data = book_array[:, l2_slice]
    
    # Split into ask and bid sides (assuming symmetric layout)
    mid_idx = book_data.shape[1] // 2
    ask_side = book_data[:, :mid_idx]  # first half (negative levels)
    bid_side = book_data[:, mid_idx:]  # second half (positive levels)
    
    # Calculate total volumes (sum of absolute values for each side)
    ask_volumes = np.sum(np.abs(ask_side), axis=1)
    bid_volumes = np.sum(np.abs(bid_side), axis=1)
    
    return bid_volumes, ask_volumes


def interactive_market_impact_plot(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused for mid now
    x,                    # time axis (len T)
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    beta_theory=0.5,      # theoretical slope
    tick_size=100,        # tick size for price conversion
):
    """
    • Q accumulates ONLY our insertions (from zero).
    • V_exp: cumulative market executions (event_type==4) from index hist_steps to (idx-1).
    • Impact = |VWAP_inserted - reference_price|.
    • Absolute mid reconstructed from cumulative Δmid (book[:,0]), anchored to ref price at first insertion.
    • In log–log panel: raw log values without normalization.
      - Allowed points (used for fit): colored
      - Zero-impact points: grey, excluded from fit
    • Theoretical line: passes through fixed intercept with slope beta_theory.
    • Prices converted from ticks to dollars using tick_size.
    • 4th graph: Evolution of total available volume on ask and bid sides.
    """

    # Accept DataFrame inputs -> dict[int] -> np.array
    if isinstance(b_seq_inp, pd.DataFrame):
        b_seq_inp = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    if isinstance(msg_seq_raw, pd.DataFrame):
        msg_seq_raw = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}

    # --- UI ---
    id_dd       = widgets.Dropdown(options=sorted(b_seq_inp.keys()), description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev    = widgets.Button(description="←")
    btn_next    = widgets.Button(description="→")
    msg_box     = widgets.HTML()
    coeff_box   = widgets.HTML()

    if 79 in b_seq_inp:
        id_dd.value = 79

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=["Book state (ΔL2)", "Mid (absolute)", "Market Impact (log–log)", "Book Volumes (Ask/Bid)"]
    )
    fig.update_layout(width=1400, height=1040, showlegend=True, template='plotly_white')
    fig_widget = go.FigureWidget(fig)

    # raw message column indices (adjust if needed)
    EVENT_TYPE_COL = 1
    DIRECTION_COL  = 2
    PRICE_COL      = 3   # absolute price in ticks
    REL_COL        = 4
    SIZE_COL       = 5

    def update_slider_range(*_):
        arr = b_seq_inp[id_dd.value]
        time_slider.min = 1
        time_slider.max = arr.shape[0] - 1
        time_slider.value = min(551, time_slider.max)

    def update_plot(*_):
        sample_id = id_dd.value
        t = time_slider.value

        book_array = b_seq_inp[sample_id]      # [T, ...]
        messages   = msg_seq_raw[sample_id]    # [T, num_fields]
        T = len(messages)

        # Get day data for this sample from sample_day_map
        sample_row = sample_day_map[sample_day_map['sample_id'] == sample_id]
        if sample_row.empty:
            coeff_box.value = f"<b>Sample {sample_id} not found in sample_day_map.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return
        
        highest_price = sample_row.iloc[0]['highest_price']
        lowest_price = sample_row.iloc[0]['lowest_price']
        execution_sum = sample_row.iloc[0]['execution_sum']
        
        # (1) Book viz (ΔL2 slices)
        # l2_slice = slice(240, 263)  # adjust to your layout
        # l2_slice = slice(20, 800)  # adjust to your layout
        l2_slice = slice(0, 1000)  # adjust to your layout
        book_prev = book_array[t-1, l2_slice]
        book_now  = book_array[t,   l2_slice]
        book_diff = np.abs(book_now) - np.abs(book_prev)
        x_lvls    = np.arange(len(book_prev)) - len(book_prev)//2
        book_cols = ['orange' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in book_diff]

        # (2) Insertion positions
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_box.value = "<b>No valid insertions.</b>"
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.layout.shapes = []
            return

        # (3) Reference absolute price at first insertion (convert from ticks to dollars)
        ref_idx = valid_insertions[0]
        reference_price = float(messages[ref_idx, PRICE_COL]) / tick_size

        # (4) Absolute mid reconstruction (for middle chart and eta calculation)
        delta_mid = book_array[:, 0].astype(float) / tick_size  # Δmid_t in dollars
        cum_delta_mid   = np.cumsum(delta_mid)
        mid_abs   = reference_price + (cum_delta_mid - cum_delta_mid[ref_idx])
        mid_series_abs  = mid_abs

        # (5) Calculate impact and market volume using helper functions - get x and y for plotting
        # Convert prices to dollars in the helper functions
        messages_dollars = messages.copy().astype(float)
        messages_dollars[:, PRICE_COL] /= tick_size
        
        impact, vwap_series, Q_cum, log_imp = calculate_impact(messages_dollars, valid_insertions, reference_price)
        V_exp, log_qv = calculate_market_volume(messages_dollars, hist_steps, valid_insertions, execution_sum)

        # (6) Calculate book volumes for 4th graph
        bid_volumes, ask_volumes = calculate_book_volumes(book_array)

        # (7) Calculate eta for display using highest_price and lowest_price from day data
        H = highest_price / tick_size  # Convert to dollars
        L = lowest_price / tick_size   # Convert to dollars
        print(f"DEBUG eta calculation: H={H:.10f}, L={L:.10f}")
        
        if H > L:
            ln_hl = np.log(H / L)
            print(f"DEBUG eta calculation: ln(H/L)={ln_hl:.10f}")
            eta_day = ln_hl / 0.8325546
            print(f"DEBUG eta calculation: eta_day = {ln_hl:.10f} / 0.8325546 = {eta_day:.10f}")
            print(f"DEBUG ln(eta) calculation: ln(eta_day) = {np.log(eta_day):.10f}")
        else:
            eta_day = 1e-12
            print(f"DEBUG eta calculation: H <= L, using eta_day = {eta_day:.10f}")

        # (8) Determine which points to use for fitting
        tol = 1e-12
        mask_zero = impact <= tol
        mask_pos  = ~mask_zero

        # (9) Fit on allowed points
        used_x_raw = log_qv[mask_pos]
        used_y_raw = log_imp[mask_pos]

        if used_x_raw.size >= 2:
            # Fit in raw log space
            # Calculate fixed intercept based on high/low of midprices
            fixed_intercept = np.log(eta_day)
            
            # Fit regression with fixed intercept
            # y = fixed_intercept + beta * x, so we solve for beta using: beta = mean((y - fixed_intercept) / x)
            adjusted_y = used_y_raw - fixed_intercept
            beta_hat = float(np.mean(adjusted_y / used_x_raw))
            alpha_hat = fixed_intercept

            # Calculate min/max and deltas for allowed points only
            x_min, x_max = float(used_x_raw.min()), float(used_x_raw.max())
            y_min, y_max = float(used_y_raw.min()), float(used_y_raw.max())
            x_delta = x_max - x_min
            y_delta = y_max - y_min

            # Theoretical line in raw log space: y = fixed_intercept + beta_theory * x
            n_used = used_x_raw.shape[0]; n_total = len(log_qv)
            coeff_box.value = f"""
            <div style="padding-left:20px; font-family:monospace">
                <h4>Market Impact (log–log; allowed points only) - Sample {sample_id}</h4>
                <p><b>Fit:</b> log(Impact) = <b>{alpha_hat:.10f}</b> + <b>{beta_hat:.10f}</b> · log(Q/V_exp)</p>
                <p><b>Theory:</b> log(Impact) = {fixed_intercept:.10f} + {beta_theory:.2f} · log(Q/V_exp)</p>
                <p><b>Eta:</b> {eta_day:.10f}</p>
                <p><b>H:</b> ${H:.2f}, <b>L:</b> ${L:.2f}, <b>Execution Sum:</b> {execution_sum:.0f}</p>
                <p>Used points: {n_used} / {n_total}</p>
                <p><b>X range:</b> [{x_min:.10f}, {x_max:.10f}] (Δ={x_delta:.10f})</p>
                <p><b>Y range:</b> [{y_min:.10f}, {y_max:.10f}] (Δ={y_delta:.10f})</p>
            </div>
            """

            # Line ranges for plotting
            xspan = np.linspace(float(used_x_raw.min()), float(used_x_raw.max()), 100)
            fit_y = alpha_hat + beta_hat * xspan
            th_y  = fixed_intercept + beta_theory * xspan

        else:
            xspan = np.array([]); fit_y = np.array([]); th_y = np.array([])
            coeff_box.value = f"""
            <div style="padding-left:20px; font-family:monospace">
                <h4>Market Impact (log–log) - Sample {sample_id}</h4>
                <p><b>Fit:</b> not enough non-zero impact points.</p>
                <p><b>Eta:</b> {eta_day:.10f}</p>
                <p><b>H:</b> ${H:.2f}, <b>L:</b> ${L:.2f}, <b>Execution Sum:</b> {execution_sum:.0f}</p>
            </div>
            """

        # Labels 1..N
        insert_labels = [str(i) for i, _ in enumerate(valid_insertions, start=1)]

        # (10) Draw
        with fig_widget.batch_update():
            fig_widget.data = []
            fig_widget.layout.shapes = []

            # (1) L2 book bars
            fig_widget.add_bar(x=x_lvls, y=book_prev, row=1, col=1, marker_color='orange', name="Prev L2")
            fig_widget.add_bar(x=x_lvls, y=book_now,  row=1, col=1, marker_color=book_cols, name="Now L2")

            # (2) mid (absolute)
            fig_widget.add_scatter(x=np.arange(T), y=mid_series_abs, mode='lines',
                                   row=1, col=2, line=dict(width=1), name="Mid (abs)")
            fig_widget.add_shape(type="line", x0=t, x1=t,
                                 y0=float(np.nanmin(mid_series_abs)),
                                 y1=float(np.nanmax(mid_series_abs)),
                                 line=dict(color="green", width=2), xref="x2", yref="y2")

            # insertion markers on mid
            insert_dots_x = [p for p in valid_insertions if p < len(mid_series_abs)]
            insert_dots_y = [mid_series_abs[p] for p in insert_dots_x]
            fig_widget.add_scatter(
                x=insert_dots_x, y=insert_dots_y, mode='markers',
                marker=dict(size=7, symbol='circle'),
                text=insert_labels[:len(insert_dots_x)],
                hovertemplate="Insertion %{text}<extra></extra>",
                row=1, col=2, name="Insert marks"
            )

            # (3) raw log–log points & lines
            if xspan.size > 0:
                # allowed (impact>0)
                fig_widget.add_scatter(
                    x=log_qv[mask_pos], y=log_imp[mask_pos], mode='markers',
                    marker=dict(size=8),
                    text=[lbl for lbl, m in zip(insert_labels, mask_pos) if m],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=2, col=1, name="Points (allowed)"
                )
                # zero-impact -> grey
                if np.any(~mask_pos):
                    fig_widget.add_scatter(
                        x=log_qv[~mask_pos], y=log_imp[~mask_pos], mode='markers',
                        marker=dict(size=8, color='grey'),
                        text=[lbl for lbl, m in zip(insert_labels, ~mask_pos) if m],
                        hovertemplate="Insertion %{text} (zero-impact)<extra></extra>",
                        row=2, col=1, name="Zero-impact (grey)"
                    )
                # fitted & theoretical lines
                fig_widget.add_scatter(x=xspan, y=fit_y, mode='lines',
                                       line=dict(dash='dot', width=2), row=2, col=1, name="Fit")
                fig_widget.add_scatter(x=xspan, y=th_y, mode='lines',
                                       line=dict(dash='dash', width=2), row=2, col=1, name=f"Theoretical (β={beta_theory:.2f})")

                fig_widget.update_xaxes(title="log(Q / V_exp)", row=2, col=1)
                fig_widget.update_yaxes(title="log(Impact)",   row=2, col=1)
            else:
                fig_widget.update_xaxes(title="log(Q / V_exp)", row=2, col=1)
                fig_widget.update_yaxes(title="log(Impact)",   row=2, col=1)

            # (4) Book volumes evolution
            fig_widget.add_scatter(x=np.arange(T), y=bid_volumes, mode='lines',
                                   line=dict(color='blue', width=1), row=2, col=2, name="Bid Volume")
            fig_widget.add_scatter(x=np.arange(T), y=ask_volumes, mode='lines',
                                   line=dict(color='red', width=1), row=2, col=2, name="Ask Volume")
            
            # Current time vertical line for volumes
            fig_widget.add_shape(type="line", x0=t, x1=t,
                                 y0=float(min(np.nanmin(bid_volumes), np.nanmin(ask_volumes))),
                                 y1=float(max(np.nanmax(bid_volumes), np.nanmax(ask_volumes))),
                                 line=dict(color="green", width=2), xref="x4", yref="y4")

            # insertion markers on volumes
            insert_dots_x_vol = [p for p in valid_insertions if p < len(bid_volumes)]
            insert_dots_y_bid = [bid_volumes[p] for p in insert_dots_x_vol]
            insert_dots_y_ask = [ask_volumes[p] for p in insert_dots_x_vol]
            
            fig_widget.add_scatter(
                x=insert_dots_x_vol, y=insert_dots_y_bid, mode='markers',
                marker=dict(size=7, symbol='circle', color='blue'),
                text=insert_labels[:len(insert_dots_x_vol)],
                hovertemplate="Insertion %{text} - Bid<extra></extra>",
                row=2, col=2, name="Insert marks (Ask)"
            )
            fig_widget.add_scatter(
                x=insert_dots_x_vol, y=insert_dots_y_ask, mode='markers',
                marker=dict(size=7, symbol='circle', color='red'),
                text=insert_labels[:len(insert_dots_x_vol)],
                hovertemplate="Insertion %{text} - Ask<extra></extra>",
                row=2, col=2, name="Insert marks (Bid)"
            )
            
            fig_widget.update_xaxes(title="Time", row=2, col=2)
            fig_widget.update_yaxes(title="Volume", row=2, col=2)

            # highlight current insertion
            if xspan.size > 0 and t in valid_insertions:
                i_sel = valid_insertions.index(t)
                x_sel = log_qv[i_sel]
                y_sel = log_imp[i_sel]
                fig_widget.add_scatter(
                    x=[x_sel], y=[y_sel], mode='markers',
                    marker=dict(color='red', size=12),
                    text=[insert_labels[i_sel]],
                    hovertemplate="Insertion %{text}<extra></extra>",
                    row=2, col=1, name="Current"
                )

        # raw message info @ t (display prices in dollars)
        m = messages[t].astype(int)
        price_dollars = m[PRICE_COL] / tick_size
        event_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        direction_map = {1: "Buy", 0: "Sell"}
        msg_box.value = (
            f"<b>{event_map.get(m[EVENT_TYPE_COL], '?')} • {direction_map.get(m[DIRECTION_COL], '?')} "
            f"• abs=${price_dollars:.2f} • rel={m[REL_COL]} • size={m[SIZE_COL]}</b><br>raw: {m.tolist()}"
        )

    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1

    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1

    id_dd.observe(lambda _: update_slider_range(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    update_slider_range()
    update_plot()

    display(widgets.HBox([id_dd, btn_prev, btn_next, time_slider]))
    display(widgets.HBox([fig_widget, coeff_box]))
    display(msg_box)


In [ ]:
# experiment_name = 'exp_89_20250828_194553_gen_sell_1024'  #       exp_85_20250823_020427_1024
# # filtration_name = 'samples_after_book_filtration_buy_1024_446'

CONFIG_PATH = f"/app/data_saved/{buy_exp_name}/used_config.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)
num_insertions      = config["num_insertions"]
num_coolings        = config["num_coolings"]
midprice_step_size  = config["midprice_step_size"]
hist_msgs           = config["n_messages"]
n_gen_msgs          = config["n_gen_msgs"]

hist_steps = hist_msgs // midprice_step_size       # 500
gen_steps = n_gen_msgs // midprice_step_size     # 50
gen_block = gen_steps + 1                        # 51

# x is just numbers from 1 to max len (all_series)
x = list(range(1, all_series.shape[1] + 1))

In [ ]:
all_betas = interactive_market_impact_plot(b_dict_combined, m_dict_combined, all_series, x, hist_steps, gen_block, num_insertions)

In [ ]:
import numpy as np

# ---------- small utils ----------
def _as_float(a):
    return np.asarray(a, dtype=float)

def _mask_xy(x, y):
    x = _as_float(x); y = _as_float(y)
    m = np.isfinite(x) & np.isfinite(y) & (x != 0.0)
    return x[m], y[m], m

def _beta_wls_fixed_internal(x, y_adj, w):
    # Weighted regression of y_adj on x with intercept fixed at 0
    x = _as_float(x); y_adj = _as_float(y_adj); w = _as_float(w)
    m = np.isfinite(x) & np.isfinite(y_adj) & np.isfinite(w) & (x != 0) & (w > 0)
    if m.sum() < 2: return np.nan
    xw = x[m] * np.sqrt(w[m]); yw = y_adj[m] * np.sqrt(w[m])
    denom = np.dot(xw, xw)
    if denom <= 0: return np.nan
    return float(np.dot(xw, yw) / denom)

# ---------- estimators (fixed intercept y = alpha + beta * x) ----------
def _beta_ols_fixed(x, y, alpha):
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    denom = np.dot(x, x)
    if denom <= 0: return np.nan
    return float(np.dot(x, y_adj) / denom)

def _beta_huber_fixed(x, y, alpha, c=1.345, max_iter=50, tol=1e-8):
    # Huber M via IRLS (defaults chosen for ~95% Gaussian efficiency)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    beta = _beta_ols_fixed(x, y, alpha)
    if not np.isfinite(beta): beta = 0.0
    for _ in range(max_iter):
        r = y_adj - beta * x
        med = np.median(r)
        sigma = 1.4826 * np.median(np.abs(r - med)) or (np.std(r) + 1e-12)
        u = r / (sigma + 1e-12)
        w = np.ones_like(u)
        big = np.abs(u) > c
        w[big] = (c / (np.abs(u[big]) + 1e-12))
        beta_new = _beta_wls_fixed_internal(x, y_adj, w)
        if not np.isfinite(beta_new): break
        if abs(beta_new - beta) <= tol * (1.0 + abs(beta)):
            beta = beta_new; break
        beta = beta_new
    return float(beta)

def _beta_lad_fixed(x, y, alpha, iters=100, eps=1e-8):
    # LAD (L1) via IRLS: w_i = 1/max(|r_i|, eps)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    beta = _beta_ols_fixed(x, y, alpha)
    if not np.isfinite(beta): beta = 0.0
    for _ in range(iters):
        r = y_adj - beta * x
        w = 1.0 / np.maximum(np.abs(r), eps)
        beta_new = _beta_wls_fixed_internal(x, y_adj, w)
        if not np.isfinite(beta_new): break
        if abs(beta_new - beta) <= 1e-7 * (1.0 + abs(beta)):
            beta = beta_new; break
        beta = beta_new
    return float(beta)

def _beta_ratio_median(x, y, alpha, x_floor=1e-6):
    # Median of ratios with |x| floor (avoid blow-ups near 0)
    x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
    m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
    r = y_adj[m] / x[m]
    if r.size == 0: return np.nan
    return float(np.median(np.sort(r)))

def _beta_ratio_trim(x, y, alpha, x_floor=1e-6, trim=0.10):
    # Trimmed-mean of ratios (default 10% each tail) with |x| floor
    x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
    m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
    r = np.sort(y_adj[m] / x[m])
    if r.size == 0: return np.nan
    k = int(trim * r.size)
    r = r[k: r.size - k] if r.size - 2*k > 0 else r
    return float(np.mean(r))

def _beta_deming_fixed(x, y, alpha, lambda_yx=1.0):
    # Orthogonal regression with fixed intercept (through origin on y_adj)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    s_xx = np.dot(x, x) / x.size
    s_yy = np.dot(y_adj, y_adj) / x.size
    s_xy = np.dot(x, y_adj) / x.size
    if s_xy == 0.0: return np.nan
    A = s_yy - lambda_yx * s_xx
    B = 2.0 * s_xy
    disc = A*A + (B*B) * lambda_yx
    beta = (A + np.sqrt(disc)) / B
    return float(beta)

# ---------- single public entry ----------
def beta_fit(x, y, alpha, method="ols"):
    """
    Estimate beta in y = alpha + beta * x with a fixed intercept.

    Parameters
    ----------
    x, y : array-like
    alpha : float or array-like (broadcastable)
    method : {'ols','huber','lad','ratio-median','ratio-trim','deming'}

    Returns
    -------
    beta : float
    """
    m = method.lower()
    if m == "ols":
        return _beta_ols_fixed(x, y, alpha)
    elif m == "huber":
        return _beta_huber_fixed(x, y, alpha)           # c=1.345, 50 iters, tol=1e-8
    elif m == "lad":
        return _beta_lad_fixed(x, y, alpha)             # 100 iters, eps=1e-8
    elif m == "ratio-median":
        return _beta_ratio_median(x, y, alpha)          # x_floor=1e-6
    elif m == "ratio-trim":
        return _beta_ratio_trim(x, y, alpha)            # trim=10%, x_floor=1e-6
    elif m == "deming":
        return _beta_deming_fixed(x, y, alpha)          # lambda_yx=1.0
    else:
        raise ValueError(f"Unknown method '{method}'. Use one of: "
                         "ols, huber, lad, ratio-median, ratio-trim, deming.")

In [ ]:
est_method="lad"

def market_impact_dashboard_from_raw(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step (ticks)
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused for mid now
    x,                    # time axis (len T) - unused here
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    *,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True,
    tick_size=100,        # convert ticks -> dollars before calling helper funcs
):
    """
    Same UI/figure as before, but x/y are computed via your helper functions:
      - impact/log_imp from calculate_impact(...)
      - V_exp/log_qv from calculate_market_volume(...), now fed with day-level execution_sum from sample_day_map
    α is fixed to ln(eta_day), where eta_day = ln(H/L)/0.8325546 using H,L from the same sample_day_map.
    β is the fixed-intercept slope: mean((y - α) / x).
    """

    # ------------- normalize inputs -------------
    if isinstance(b_seq_inp, pd.DataFrame):
        b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    else:
        b_dict_local = b_seq_inp
    if isinstance(msg_seq_raw, pd.DataFrame):
        m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
    else:
        m_dict_local = msg_seq_raw

    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5

    eps = 1e-12
    tol = 1e-12

    # -------------------- compute x_df, y_df, coeffs_df via helpers --------------------
    def compute_tables():
        sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
        col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
        x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        coeff_rows = []

        for sid in sample_ids:
            messages_ticks = m_dict_local[sid]
            book = b_dict_local[sid]
            T = len(messages_ticks)

            # insertion schedule
            insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
            valid_insertions = [pos for pos in insertion_positions if pos < T]
            if not valid_insertions:
                coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan,
                                   "n_used": 0, "n_total": 0})
                continue

            # Reference price at first insertion (ticks -> $)
            ref_idx = valid_insertions[0]
            reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

            # -------- NEW: get day info from sample_day_map --------
            # Look up this sample_id in sample_day_map
            try:
                day_row = sample_day_map[sample_day_map['sample_id'] == sid]
                if not day_row.empty:
                    H_ticks = float(day_row.iloc[0]['highest_price'])
                    L_ticks = float(day_row.iloc[0]['lowest_price'])
                    execution_sum = float(day_row.iloc[0]['execution_sum'])
                else:
                    # Fallback if not found: infer H/L from pre-gen window and use window exec sum
                    H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                    L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                    exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                    execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
            except Exception as _e:
                # Fallback if not found: infer H/L from pre-gen window and use window exec sum
                H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

            # Convert to dollars for Parkinson eta
            H = float(H_ticks) / tick_size
            L = float(L_ticks) / tick_size
            if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
                eta_day = np.log(H / L) / 0.8325546
                alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
            else:
                eta_day = eps
                alpha_fixed = float(np.log(eta_day))

            # Convert messages to dollars for the helper functions
            messages_dollars = messages_ticks.astype(float).copy()
            messages_dollars[:, PRICE_COL] /= tick_size

            # --- use your helper functions (unchanged graphs) ---
            impact, vwap_series, Q_cum, log_imp = calculate_impact(
                messages_dollars, valid_insertions, reference_price
            )
            # -------- NEW: pass execution_sum from sample_day_map into V_exp calc --------
            V_exp, log_qv = calculate_market_volume(
                messages_dollars, hist_steps, valid_insertions, execution_sum
            )

            # fill tables per insertion (same)
            mask_zero = impact <= tol
            mask_pos  = ~mask_zero
            for j, _idx in enumerate(valid_insertions):
                col = f"ins_{j+1}"
                if mask_zero[j] or not np.isfinite(log_qv[j]) or not np.isfinite(log_imp[j]):
                    x_df.loc[sid, col] = "ZERO"
                    y_df.loc[sid, col] = "ZERO"
                else:
                    x_df.loc[sid, col] = float(log_qv[j])
                    y_df.loc[sid, col] = float(log_imp[j])

            # per-sample β with fixed intercept (same)
            used_x = log_qv[mask_pos]
            used_y = log_imp[mask_pos]
            n_used = int(used_x.size)
            n_total = int(len(valid_insertions))
            if n_used >= 2 and np.all(np.isfinite(used_x)) and np.all(np.isfinite(used_y)):
                valid_mask = (used_x != 0) & np.isfinite(used_x) & np.isfinite(used_y)
                if np.sum(valid_mask) >= 2:
                    # beta_hat = float(np.mean((used_y[valid_mask] - alpha_fixed) / used_x[valid_mask]))
                    beta_hat = beta_fit(used_x[valid_mask], used_y[valid_mask], alpha_fixed, method=est_method)                    
                else:
                    beta_hat = np.nan
            else:
                beta_hat = np.nan

            coeff_rows.append({
                "sample_id": sid,
                "alpha_hat": alpha_fixed,   # fixed ln(η_day) from sample_day_map
                "beta_hat": beta_hat,
                "n_used": n_used,
                "n_total": n_total,
            })

        coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
        return x_df, y_df, coeffs_df

    x_df, y_df, coeffs_df = compute_tables()

    # -------------------- tidy points (unchanged) --------------------
    all_ids = list(x_df.index)
    ordered_ids = [sid for sid in special_first if sid in all_ids]
    ordered_ids += [sid for sid in sorted(all_ids) if sid not in ordered_ids]
    if samples_used is not None:
        ordered_ids = ordered_ids[:samples_used]

    rows = []
    for sid in ordered_ids:
        for j, col in enumerate(x_df.columns, start=1):
            xv = x_df.loc[sid, col]
            yv = y_df.loc[sid, col]
            if isinstance(xv, (int, float, np.floating)) and isinstance(yv, (int, float, np.floating)):
                if np.isfinite(xv) and np.isfinite(yv):
                    rows.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
    points_df = pd.DataFrame(rows)
    if points_df.empty:
        print("No numeric points to plot.")
        return None, pd.DataFrame(), pd.DataFrame(), {}

    data_max_ins = int(points_df["insertion"].max())
    max_insertions = int(min(num_insertions, data_max_ins))
    a_values = np.arange(1, max_insertions + 1)

    # -------------------- histogram data (unchanged) --------------------
    betas_clean = coeffs_df["beta_hat"].replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    beta_mean = float(betas_clean.mean()) if not betas_clean.empty else np.nan

    # -------------------- figure (unchanged) --------------------
    fig = go.FigureWidget(make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "xy"}, {"type": "xy"}],
               [None,          {"type": "xy"}]],
        column_widths=[0.68, 0.32],
        row_heights=[0.55, 0.45],
        horizontal_spacing=0.07,
        vertical_spacing=0.12,
        subplot_titles=("Scatter & Global Fit", "β(a) evolution", "Distribution of β̂ across samples")
    ))

    LEGEND_MAX = 15
    palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b",
               "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#ff7f0e"]
    GREY = "rgba(0,0,0,0.25)"

    trace_meta = []
    for i, sid in enumerate(ordered_ids):
        sub = points_df[points_df["sample_id"] == sid].sort_values("insertion")
        x_vals = sub["x"].to_numpy()
        y_vals = sub["y"].to_numpy()
        ins = sub["insertion"].to_numpy()
        base_color = palette[i % len(palette)] if i != 1 else "#d62728"
        tr = go.Scatter(
            x=x_vals, y=y_vals, mode="markers",
            name=f"sample {sid} ({len(sub)}/{len(sub)} pts)",
            legendgroup=str(sid), showlegend=(i < LEGEND_MAX),
            marker=dict(size=7, color=base_color),
            hovertemplate=(
                "sample=%{customdata[0]}<br>"
                "ins=%{customdata[1]}<br>"
                "log(Q/V)=%{x:.4f}<br>"
                "log(Impact)=%{y:.4f}<extra></extra>"
            ),
            customdata=np.stack([sub["sample_id"].to_numpy(), ins], axis=1),
        )
        fig.add_trace(tr, row=1, col=1)
        trace_meta.append({"sid": sid, "ins": ins, "x": x_vals, "y": y_vals,
                           "base_color": base_color, "total": len(sub)})

    # global-fit trace (same)
    fit_trace_index = len(fig.data)
    fig.add_trace(
        go.Scatter(x=[], y=[], mode="lines", name="Global fit",
                   line=dict(dash="dash", width=2)),
        row=1, col=1
    )

    # ----- fitting helpers (unchanged, uses α_global = mean ln(η)) -----
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    def fit_for_mask(mask):
        X = points_df.loc[mask, "x"].to_numpy()
        Y = points_df.loc[mask, "y"].to_numpy()
        if len(Y) < 2:
            return np.nan, np.nan, np.nan, 0, np.array([]), np.array([])
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid_mask) < 2:
            return alpha_global, np.nan, np.nan, len(Y), np.array([]), np.array([])
        x_valid = X[valid_mask]; y_valid = Y[valid_mask]


        # ================================ #

        
        # beta = float(np.mean((y_valid - alpha_global) / x_valid))
        beta = beta_fit(x_valid, y_valid, alpha_global, method=est_method)
        

        # ================================ #
        
        y_pred = alpha_global + beta * X
        ss_tot = float(((Y - Y.mean()) ** 2).sum())
        ss_res = float(((Y - y_pred) ** 2).sum())
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        x_min, x_max = X.min(), X.max()
        x_line = np.linspace(x_min, x_max, 200)
        y_line = alpha_global + beta * x_line
        return alpha_global, beta, r2, len(Y), x_line, y_line

    # precompute β(a) & fit lines (unchanged)
    a_values = np.arange(1, max_insertions + 1)
    betas_evo = np.full_like(a_values, np.nan, dtype=float)
    fit_lines = {}
    for idx, a in enumerate(a_values):
        mask = points_df["insertion"] >= a
        alpha, beta, r2, n, x_line, y_line = fit_for_mask(mask)
        betas_evo[idx] = beta
        fit_lines[a] = (x_line, y_line, alpha, beta, r2, n)

    # β(a) and indicator (unchanged)
    beta_line_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(x=a_values, y=betas_evo, mode="lines+markers", name="β(a)"),
        row=1, col=2
    )
    beta_vline_idx = len(fig.data)
    y_min = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
    y_max = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
    fig.add_trace(
        go.Scatter(x=[a_values[0], a_values[0]], y=[y_min, y_max],
                   mode="lines", line=dict(color="green", dash="dot"), name="current a"),
        row=1, col=2
    )

    # histogram β̂ (unchanged)
    if not betas_clean.empty:
        fig.add_trace(go.Histogram(x=betas_clean.values, nbinsx=30, name="β̂"), row=2, col=2)
        fig.add_trace(go.Scatter(x=[beta_mean, beta_mean], y=[0, max(1, len(betas_clean))],
                                 mode="lines", name=f"Mean β̂ = {beta_mean:.4f}",
                                 line=dict(dash="dash")), row=2, col=2)
        fig.add_trace(go.Scatter(x=[beta_theory, beta_theory], y=[0, max(1, len(betas_clean))],
                                 mode="lines", name=f"β = {beta_theory:.2f} (theoretical)",
                                 line=dict(dash="dot")), row=2, col=2)

    # axes & layout (unchanged)
    fig.update_xaxes(title_text="log(Q / V_exp)", row=1, col=1)
    fig.update_yaxes(title_text="log(Impact)", row=1, col=1)
    fig.update_xaxes(title_text="a (insertion threshold)", row=1, col=2)
    fig.update_yaxes(title_text="β (slope)", row=1, col=2)
    fig.update_xaxes(title_text="β̂", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)

    fig.update_layout(template="plotly_white", width=1500, height=800,
                      margin=dict(t=70, r=50, b=60, l=60),
                      legend=dict(orientation="v"))

    # fit box (unchanged)
    def set_fit_annotation(text_html: str):
        fig.layout.annotations = tuple(
            a for a in (fig.layout.annotations or [])
            if getattr(a, "name", "") != "fit_box"
        )
        fig.add_annotation(
            x=0.02, y=0.98, xref="paper", yref="paper",
            text=text_html, showarrow=False, align="left",
            bordercolor="lightgray", borderwidth=1, borderpad=8,
            bgcolor="rgba(245,245,245,1)", name="fit_box"
        )

    def update_left_fit(a_val: int):
        x_line, y_line, alpha, beta, r2, n = fit_lines.get(a_val, ([], [], np.nan, np.nan, np.nan, 0))
        fig.data[fit_trace_index].x = x_line
        fig.data[fit_trace_index].y = y_line
        if show_fit and n > 0 and np.isfinite(beta):
            fit_html = (
                "<b>Market Impact Fit (Fixed Intercept)</b><br>"
                f"<b>a:</b> {a_val}<br>"
                f"<b>Model:</b> log(Impact) = <b>{alpha:.6f}</b> + <b>{beta:.6f}</b> · log(Q/V)<br>"
                f"<b>α (fixed):</b> {alpha:.6f} (mean ln(η) across samples)<br>"
                f"<b>β:</b> {beta:.6f}<br>"
                f"<b>R²:</b> {r2:.4f}<br>"
                f"<b>Used points:</b> {n}"
            )
        else:
            fit_html = f"<b>Market Impact Fit (Fixed Intercept)</b><br><b>a:</b> {a_val}<br>No active points."
        set_fit_annotation(fit_html)

    def recolor_and_refit(a_val: int):
        for t_idx, meta in enumerate(trace_meta):
            active_mask = meta["ins"] >= a_val
            colors = [meta["base_color"] if ok else GREY for ok in active_mask]
            fig.data[t_idx].marker.color = colors
            active_count = int(np.count_nonzero(active_mask))
            fig.data[t_idx].name = f"sample {meta['sid']} ({meta['total']}/{active_count} pts)"
        if show_fit:
            update_left_fit(a_val)

        y_min_local = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
        y_max_local = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
        fig.data[beta_vline_idx].x = [a_val, a_val]
        fig.data[beta_vline_idx].y = [y_min_local, y_max_local]

    # controls (unchanged)
    a_slider = widgets.IntSlider(value=1, min=1, max=max_insertions, step=1, description="a")
    prev_btn = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
    next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))

    def on_prev(_):
        if a_slider.value > a_slider.min:
            a_slider.value -= 1

    def on_next(_):
        if a_slider.value < a_slider.max:
            a_slider.value += 1

    def on_a_change(change):
        if change["name"] == "value":
            recolor_and_refit(change["new"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    a_slider.observe(on_a_change, names="value")

    # initial render
    recolor_and_refit(a_slider.value)
    display(widgets.HBox([prev_btn, next_btn, a_slider]), fig)

    controls = {"a_slider": a_slider, "prev_btn": prev_btn, "next_btn": next_btn}
    return fig, points_df, coeffs_df, controls

In [ ]:
# Call the market impact dashboard function
fig, points_df, coeffs_df, controls = market_impact_dashboard_from_raw(
    b_seq_inp=b_dict_combined,
    msg_seq_raw=m_dict_combined,
    all_series=all_series,
    x=x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True
)

In [ ]:
num_insertions

In [ ]:
est_method="lad"

# Call the market impact dashboard function
fig, points_df, coeffs_df, controls = market_impact_dashboard_from_raw(
    b_seq_inp=b_dict_combined,
    msg_seq_raw=m_dict_combined,
    all_series=all_series,
    x=x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True
)

In [ ]:
# Create beta(a) evolution plot with different regression estimations using Plotly
import plotly.graph_objects as go
import numpy as np

# Define all available estimation methods from beta_fit function
methods = ["ols", "lad", "huber", "ratio-median", "ratio-trim", "deming"]
method_labels = {
    "ols": "OLS", 
    "lad": "LAD", 
    "huber": "Huber",
    "ratio-median": "Ratio Median",
    "ratio-trim": "Ratio Trim",
    "deming": "Deming"
}
colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b", "#e377c2"]

# Create Plotly figure
fig = go.Figure()

# Calculate max_insertions from points_df
max_insertions = points_df["insertion"].max() if not points_df.empty else 10

# Calculate beta evolution for each method
a_values = np.arange(1, max_insertions + 1)

all_betas_evo = {}

for method_idx, method in enumerate(methods):
    betas_evo = np.full_like(a_values, np.nan, dtype=float)
    
    # Calculate alpha_global (same for all methods)
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0
    
    for idx, a in enumerate(a_values):
        # Filter points for current threshold
        mask = points_df["insertion"] >= a
        X = points_df.loc[mask, "x"].to_numpy()
        Y = points_df.loc[mask, "y"].to_numpy()
        
        if len(Y) >= 2:
            valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
            if np.sum(valid_mask) >= 2:
                x_valid = X[valid_mask]
                y_valid = Y[valid_mask]
                try:
                    beta = beta_fit(x_valid, y_valid, alpha_global, method=method)
                    betas_evo[idx] = beta
                except Exception:
                    betas_evo[idx] = np.nan
    
    all_betas_evo[method] = betas_evo
    # Add trace for this method
    fig.add_trace(go.Scatter(
        x=a_values, 
        y=betas_evo, 
        mode='lines+markers',
        name=f'{method_labels[method]} regression',
        line=dict(color=colors[method_idx], width=2),
        marker=dict(size=6, color=colors[method_idx])
    ))

# Add theoretical beta line
fig.add_trace(go.Scatter(
    x=[a_values[0], a_values[-1]], 
    y=[0.5, 0.5],
    mode='lines',
    name='Theoretical β = 0.5',
    line=dict(color='black', dash='dash', width=2)
))

# Customize layout
fig.update_layout(
    title=dict(
        text='Beta Evolution with Different Regression Methods',
        font=dict(size=16, family="Arial Black")
    ),
    xaxis=dict(
        title='a (insertion threshold)',
        title_font=dict(size=14),
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        title='β (slope)',
        title_font=dict(size=14),
        tickfont=dict(size=12),
        range=[0.0, 1.0]
    ),
    template='plotly_white',
    width=800,
    height=800,
    legend=dict(
        font=dict(size=12),
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    showlegend=True
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

fig.show()


In [ ]:
# Create an interactive plotly widget to switch between insertion points with all estimation methods shown simultaneously
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# First, prepare data for all insertions
# Group by sample_id and create insertion numbers based on order
points_df_with_insertion = points_df.copy()
points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
max_insertions = points_df_with_insertion['insertion_number'].max()

# Calculate alpha_global
alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

# Available estimation methods
estimation_methods = ['ols', 'huber', 'lad', 'ratio-median', 'ratio-trim', 'deming']
method_colors = {
    'ols': 'red',
    'huber': 'blue', 
    'lad': 'orange',
    'ratio-median': 'purple',
    'ratio-trim': 'brown',
    'deming': 'pink'
}

# Prepare data for each insertion with all methods
insertion_data_dict = {}
for insertion_num in range(1, max_insertions + 1):
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        X = insertion_data["x"].to_numpy()
        Y = insertion_data["y"].to_numpy()
        sample_ids = insertion_data["sample_id"].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        x_valid = X[valid_mask]
        y_valid = Y[valid_mask]
        
        # Calculate fitted beta for each method
        beta_fitted_dict = {}
        for method in estimation_methods:
            if len(x_valid) >= 2:
                beta_fitted_dict[method] = beta_fit(x_valid, y_valid, alpha_global, method=method)
            else:
                beta_fitted_dict[method] = np.nan
        
        insertion_data_dict[insertion_num] = {
            'x_all': X,
            'y_all': Y,
            'sample_ids': sample_ids,
            'x_valid': x_valid,
            'y_valid': y_valid,
            'beta_fitted_dict': beta_fitted_dict,
            'n_points': len(X),
            'n_valid': len(x_valid)
        }

# Create FigureWidget for interactive updates
fig = go.FigureWidget()

# Create widgets
insertion_dropdown = widgets.Dropdown(
    options=[(f'Insertion {i}', i) for i in range(1, max_insertions + 1)],
    value=1,
    description='Insertion:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Arrow control buttons
prev_button = widgets.Button(
    description='◀ Previous',
    button_style='info',
    layout=widgets.Layout(width='100px')
)

next_button = widgets.Button(
    description='Next ▶',
    button_style='info',
    layout=widgets.Layout(width='100px')
)

# Create output widget for statistics
stats_output = widgets.Output()

def update_plot(insertion_num):
    if insertion_num in insertion_data_dict:
        data = insertion_data_dict[insertion_num]
        
        # Clear existing traces
        with fig.batch_update():
            fig.data = []
            
            # Separate points by buy/sell based on sample_id
            buy_mask = data['sample_ids'] < 1000000
            sell_mask = data['sample_ids'] >= 1000000
            
            # Add scatter plot for buy orders (blue/green)
            if np.any(buy_mask):
                fig.add_scatter(
                    x=data['x_all'][buy_mask],
                    y=data['y_all'][buy_mask],
                    mode='markers',
                    name=f'Buy orders (n={np.sum(buy_mask)})',
                    marker=dict(size=8, opacity=0.7, color='green'),
                    hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra></extra>'
                )
            
            # Add scatter plot for sell orders (red)
            if np.any(sell_mask):
                fig.add_scatter(
                    x=data['x_all'][sell_mask],
                    y=data['y_all'][sell_mask],
                    mode='markers',
                    name=f'Sell orders (n={np.sum(sell_mask)})',
                    marker=dict(size=8, opacity=0.7, color='red'),
                    hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra></extra>'
                )
            
            # Add fitted lines for all methods
            if len(data['x_valid']) >= 2 and len(data['x_all']) > 0:
                x_range = np.linspace(data['x_all'].min(), data['x_all'].max(), 100)
                
                for method in estimation_methods:
                    beta_fitted = data['beta_fitted_dict'][method]
                    if np.isfinite(beta_fitted):
                        y_fitted = alpha_global + beta_fitted * x_range
                        fig.add_scatter(
                            x=x_range,
                            y=y_fitted,
                            mode='lines',
                            name=f'{method.upper()}: β = {beta_fitted:.6f}',
                            line=dict(color=method_colors[method], dash='dash', width=2)
                        )
            
            # Add theoretical line
            beta_theory = 0.5
            if len(data['x_all']) > 0:
                x_range = np.linspace(data['x_all'].min(), data['x_all'].max(), 100)
                y_theory = alpha_global + beta_theory * x_range
                fig.add_scatter(
                    x=x_range,
                    y=y_theory,
                    mode='lines',
                    name=f'Theoretical: β = {beta_theory:.6f}',
                    line=dict(color='black', dash='dot', width=3)
                )
            
            # Update layout
            fig.update_layout(
                title=f'Market Impact: Insertion {insertion_num} (All Estimation Methods)',
                xaxis_title='log(Q / V_exp)',
                yaxis_title='log(Impact)',
                template='plotly_white',
                width=1200,
                height=700,
                legend=dict(
                    font=dict(size=11),
                    orientation="v",
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=0.01
                ),
                showlegend=True
            )
            
            # Add grid
            fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
            fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        
        # Update statistics output
        with stats_output:
            stats_output.clear_output()
            print(f"Insertion {insertion_num} Statistics:")
            print(f"Alpha (fixed intercept): {alpha_global:.6f}")
            print(f"Theoretical Beta: {beta_theory:.6f}")
            print(f"Total points: {data['n_points']}")
            print(f"Valid points: {data['n_valid']}")
            print(f"Buy orders: {np.sum(data['sample_ids'] < 1000000)}")
            print(f"Sell orders: {np.sum(data['sample_ids'] >= 1000000)}")
            print()
            print("Fitted Beta values by method:")
            for method in estimation_methods:
                beta_fitted = data['beta_fitted_dict'][method]
                print(f"  {method.upper():12}: β = {beta_fitted:.6f}")

# Connect widgets to update function
def on_dropdown_change(change):
    update_plot(insertion_dropdown.value)

def on_prev_click(b):
    current_val = insertion_dropdown.value
    if current_val > 1:
        insertion_dropdown.value = current_val - 1

def on_next_click(b):
    current_val = insertion_dropdown.value
    if current_val < max_insertions:
        insertion_dropdown.value = current_val + 1

insertion_dropdown.observe(on_dropdown_change, names='value')
prev_button.on_click(on_prev_click)
next_button.on_click(on_next_click)

# Initialize with first insertion
update_plot(1)

# Create control panel layout
controls_row = widgets.HBox([prev_button, insertion_dropdown, next_button])
controls = widgets.VBox([controls_row])

# Display widgets and plot
display(widgets.VBox([controls, stats_output, fig]))


In [ ]:
# Create an interactive plotly widget to show buy and sell samples together on one plot
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# First, prepare data for all samples
# Get unique sample IDs and separate buy/sell
unique_samples = sorted(points_df['sample_id'].unique())
buy_samples = [s for s in unique_samples if s < 1000000]
sell_samples = [s for s in unique_samples if s >= 1000000]

# Calculate alpha_global
alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

# Prepare data for each sample with detailed message info
sample_data_dict = {}
for sample_id in unique_samples:
    sample_data = points_df[points_df['sample_id'] == sample_id].copy()
    
    if len(sample_data) > 0:
        # Sort by insertion order (assuming data is already ordered)
        sample_data = sample_data.sort_index()
        
        X = sample_data["x"].to_numpy()
        Y = sample_data["y"].to_numpy()
        
        # Get detailed message info for hover
        hover_info = []
        volumes = []
        message_ids = []
        insertion_numbers = []
        
        for idx, (_, row) in enumerate(sample_data.iterrows()):
            insertion_num = idx + 1
            insertion_numbers.append(insertion_num)
            
            if sample_id in m_dict_combined:
                messages = m_dict_combined[sample_id]
                # Find insertion points (messages with id 77777777)
                insertion_points = messages[messages[:, 0] == 77777777]
                
                # Get the volume for this insertion (idx because 0-indexed)
                if idx < len(insertion_points):
                    volume = insertion_points[idx, 5]  # 5th index is volume
                    msg_id = insertion_points[idx, 0]  # 0th index is message id
                    volumes.append(volume)
                    message_ids.append(msg_id)
                    
                    # Create detailed hover text
                    hover_text = (f"Sample ID: {sample_id}<br>"
                                f"Insertion: {insertion_num}<br>"
                                f"Message ID: {msg_id}<br>"
                                f"Volume: {volume}<br>"
                                f"x (log(Q/V_exp)): {X[idx]:.6f}<br>"
                                f"y (log(Impact)): {Y[idx]:.6f}")
                    hover_info.append(hover_text)
                else:
                    volumes.append(np.nan)
                    message_ids.append(np.nan)
                    hover_info.append(f"Sample ID: {sample_id}<br>Insertion: {insertion_num}<br>No insertion point data")
            else:
                volumes.append(np.nan)
                message_ids.append(np.nan)
                hover_info.append(f"Sample ID: {sample_id}<br>Insertion: {insertion_num}<br>No message data")
        
        # Filter out invalid points for fitting
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        x_valid = X[valid_mask]
        y_valid = Y[valid_mask]
        
        # Calculate fitted beta for OLS only
        beta_fitted_ols = np.nan
        if len(x_valid) >= 2:
            beta_fitted_ols = beta_fit(x_valid, y_valid, alpha_global, method='ols')
        
        # Determine if this is a buy or sell order
        order_type = "Buy" if sample_id < 1000000 else "Sell"
        
        sample_data_dict[sample_id] = {
            'x_all': X,
            'y_all': Y,
            'volumes': np.array(volumes),
            'message_ids': np.array(message_ids),
            'insertion_numbers': np.array(insertion_numbers),
            'hover_info': hover_info,
            'x_valid': x_valid,
            'y_valid': y_valid,
            'beta_fitted_ols': beta_fitted_ols,
            'n_points': len(X),
            'n_valid': len(x_valid),
            'order_type': order_type
        }

# Create FigureWidget for interactive updates
fig = go.FigureWidget()

# Create widgets - dropdown for buy samples only
buy_dropdown = widgets.Dropdown(
    options=[(f'Buy Sample {sample_id}', sample_id) for sample_id in buy_samples],
    value=buy_samples[0] if buy_samples else None,
    description='Buy Sample:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

sell_dropdown = widgets.Dropdown(
    options=[(f'Sell Sample {sample_id}', sample_id) for sample_id in sell_samples],
    value=sell_samples[0] if sell_samples else None,
    description='Sell Sample:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Arrow control buttons for buy samples
prev_buy_button = widgets.Button(
    description='◀ Prev Buy',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

next_buy_button = widgets.Button(
    description='Next Buy ▶',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

# Arrow control buttons for sell samples
prev_sell_button = widgets.Button(
    description='◀ Prev Sell',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

next_sell_button = widgets.Button(
    description='Next Sell ▶',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

# Create output widget for statistics
stats_output = widgets.Output()

def update_plot(buy_sample_id, sell_sample_id):
    # Clear existing traces
    with fig.batch_update():
        fig.data = []
        
        # Collect all valid points from both active samples for combined fitting
        combined_x_valid = []
        combined_y_valid = []
        
        # Add buy sample data
        if buy_sample_id and buy_sample_id in sample_data_dict:
            buy_data = sample_data_dict[buy_sample_id]
            combined_x_valid.extend(buy_data['x_valid'])
            combined_y_valid.extend(buy_data['y_valid'])
            
            # Add scatter plot for buy sample
            fig.add_scatter(
                x=buy_data['x_all'],
                y=buy_data['y_all'],
                mode='markers+lines',
                name=f'Buy {buy_sample_id} (n={buy_data["n_points"]})',
                marker=dict(size=10, opacity=0.8, color='green'),
                line=dict(color='green', width=2, dash='dot'),
                hovertemplate='%{text}<extra></extra>',
                text=buy_data['hover_info']
            )
            
            # Add fitted line for buy sample OLS
            if len(buy_data['x_valid']) >= 2 and len(buy_data['x_all']) > 0:
                x_range = np.linspace(buy_data['x_all'].min(), buy_data['x_all'].max(), 100)
                beta_fitted = buy_data['beta_fitted_ols']
                if np.isfinite(beta_fitted):
                    y_fitted = alpha_global + beta_fitted * x_range
                    fig.add_scatter(
                        x=x_range,
                        y=y_fitted,
                        mode='lines',
                        name=f'Buy OLS: β = {beta_fitted:.6f}',
                        line=dict(color='darkgreen', dash='dash', width=2)
                    )
        
        # Add sell sample data
        if sell_sample_id and sell_sample_id in sample_data_dict:
            sell_data = sample_data_dict[sell_sample_id]
            combined_x_valid.extend(sell_data['x_valid'])
            combined_y_valid.extend(sell_data['y_valid'])
            
            # Add scatter plot for sell sample
            fig.add_scatter(
                x=sell_data['x_all'],
                y=sell_data['y_all'],
                mode='markers+lines',
                name=f'Sell {sell_sample_id} (n={sell_data["n_points"]})',
                marker=dict(size=10, opacity=0.8, color='red'),
                line=dict(color='red', width=2, dash='dot'),
                hovertemplate='%{text}<extra></extra>',
                text=sell_data['hover_info']
            )
            
            # Add fitted line for sell sample OLS
            if len(sell_data['x_valid']) >= 2 and len(sell_data['x_all']) > 0:
                x_range = np.linspace(sell_data['x_all'].min(), sell_data['x_all'].max(), 100)
                beta_fitted = sell_data['beta_fitted_ols']
                if np.isfinite(beta_fitted):
                    y_fitted = alpha_global + beta_fitted * x_range
                    fig.add_scatter(
                        x=x_range,
                        y=y_fitted,
                        mode='lines',
                        name=f'Sell OLS: β = {beta_fitted:.6f}',
                        line=dict(color='darkred', dash='dash', width=2)
                    )
        
        # Add combined fitted line for both active samples
        if len(combined_x_valid) >= 2:
            combined_x_valid = np.array(combined_x_valid)
            combined_y_valid = np.array(combined_y_valid)
            beta_combined = beta_fit(combined_x_valid, combined_y_valid, alpha_global, method='ols')
            
            if np.isfinite(beta_combined):
                # Get combined x range from both samples
                all_x = []
                if buy_sample_id and buy_sample_id in sample_data_dict:
                    all_x.extend(sample_data_dict[buy_sample_id]['x_all'])
                if sell_sample_id and sell_sample_id in sample_data_dict:
                    all_x.extend(sample_data_dict[sell_sample_id]['x_all'])
                
                if len(all_x) > 0:
                    x_range = np.linspace(min(all_x), max(all_x), 100)
                    y_combined = alpha_global + beta_combined * x_range
                    fig.add_scatter(
                        x=x_range,
                        y=y_combined,
                        mode='lines',
                        name=f'Combined OLS: β = {beta_combined:.6f}',
                        line=dict(color='purple', dash='solid', width=3)
                    )
        
        # Add theoretical line
        beta_theory = 0.5
        # Get combined x range from both samples
        all_x = []
        if buy_sample_id and buy_sample_id in sample_data_dict:
            all_x.extend(sample_data_dict[buy_sample_id]['x_all'])
        if sell_sample_id and sell_sample_id in sample_data_dict:
            all_x.extend(sample_data_dict[sell_sample_id]['x_all'])
        
        if len(all_x) > 0:
            x_range = np.linspace(min(all_x), max(all_x), 100)
            y_theory = alpha_global + beta_theory * x_range
            fig.add_scatter(
                x=x_range,
                y=y_theory,
                mode='lines',
                name=f'Theoretical: β = {beta_theory:.6f}',
                line=dict(color='black', dash='dot', width=3)
            )
        
        # Update layout
        fig.update_layout(
            title=f'Market Impact: Buy Sample {buy_sample_id} vs Sell Sample {sell_sample_id}',
            xaxis_title='log(Q / V_exp)',
            yaxis_title='log(Impact)',
            template='plotly_white',
            width=1200,
            height=700,
            legend=dict(
                font=dict(size=11),
                orientation="v",
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=0.01
            ),
            showlegend=True
        )
        
        # Add grid
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
    
    # Update statistics output
    with stats_output:
        stats_output.clear_output()
        
        # Buy sample statistics
        if buy_sample_id and buy_sample_id in sample_data_dict:
            buy_data = sample_data_dict[buy_sample_id]
            print(f"Buy Sample {buy_sample_id} Statistics:")
            print(f"  OLS Beta: {buy_data['beta_fitted_ols']:.6f}")
            print(f"  Total insertion points: {buy_data['n_points']}")
            print(f"  Valid points for fitting: {buy_data['n_valid']}")
            
            valid_volumes = buy_data['volumes'][np.isfinite(buy_data['volumes'])]
            if len(valid_volumes) > 0:
                print(f"  Volume range: {np.min(valid_volumes):.0f} - {np.max(valid_volumes):.0f}")
                print(f"  Mean volume: {np.mean(valid_volumes):.2f}")
        
        print()
        
        # Sell sample statistics
        if sell_sample_id and sell_sample_id in sample_data_dict:
            sell_data = sample_data_dict[sell_sample_id]
            print(f"Sell Sample {sell_sample_id} Statistics:")
            print(f"  OLS Beta: {sell_data['beta_fitted_ols']:.6f}")
            print(f"  Total insertion points: {sell_data['n_points']}")
            print(f"  Valid points for fitting: {sell_data['n_valid']}")
            
            valid_volumes = sell_data['volumes'][np.isfinite(sell_data['volumes'])]
            if len(valid_volumes) > 0:
                print(f"  Volume range: {np.min(valid_volumes):.0f} - {np.max(valid_volumes):.0f}")
                print(f"  Mean volume: {np.mean(valid_volumes):.2f}")
        
        print()
        
        # Combined statistics
        combined_x_valid = []
        combined_y_valid = []
        if buy_sample_id and buy_sample_id in sample_data_dict:
            combined_x_valid.extend(sample_data_dict[buy_sample_id]['x_valid'])
            combined_y_valid.extend(sample_data_dict[buy_sample_id]['y_valid'])
        if sell_sample_id and sell_sample_id in sample_data_dict:
            combined_x_valid.extend(sample_data_dict[sell_sample_id]['x_valid'])
            combined_y_valid.extend(sample_data_dict[sell_sample_id]['y_valid'])
        
        if len(combined_x_valid) >= 2:
            combined_x_valid = np.array(combined_x_valid)
            combined_y_valid = np.array(combined_y_valid)
            beta_combined = beta_fit(combined_x_valid, combined_y_valid, alpha_global, method='ols')
            print(f"Combined Active Samples Statistics:")
            print(f"  Combined OLS Beta: {beta_combined:.6f}")
            print(f"  Total combined valid points: {len(combined_x_valid)}")
            print()
        
        print(f"Global Statistics:")
        print(f"  Alpha (fixed intercept): {alpha_global:.6f}")
        print(f"  Theoretical Beta: {beta_theory:.6f}")

# Connect widgets to update function
def on_dropdown_change(change):
    update_plot(buy_dropdown.value, sell_dropdown.value)

def on_prev_buy_click(b):
    current_val = buy_dropdown.value
    current_idx = buy_samples.index(current_val)
    if current_idx > 0:
        buy_dropdown.value = buy_samples[current_idx - 1]

def on_next_buy_click(b):
    current_val = buy_dropdown.value
    current_idx = buy_samples.index(current_val)
    if current_idx < len(buy_samples) - 1:
        buy_dropdown.value = buy_samples[current_idx + 1]

def on_prev_sell_click(b):
    current_val = sell_dropdown.value
    current_idx = sell_samples.index(current_val)
    if current_idx > 0:
        sell_dropdown.value = sell_samples[current_idx - 1]

def on_next_sell_click(b):
    current_val = sell_dropdown.value
    current_idx = sell_samples.index(current_val)
    if current_idx < len(sell_samples) - 1:
        sell_dropdown.value = sell_samples[current_idx + 1]

buy_dropdown.observe(on_dropdown_change, names='value')
sell_dropdown.observe(on_dropdown_change, names='value')
prev_buy_button.on_click(on_prev_buy_click)
next_buy_button.on_click(on_next_buy_click)
prev_sell_button.on_click(on_prev_sell_click)
next_sell_button.on_click(on_next_sell_click)

# Initialize with first samples
if buy_samples and sell_samples:
    update_plot(buy_samples[0], sell_samples[0])

# Create control panel layout
buy_controls_row = widgets.HBox([prev_buy_button, buy_dropdown, next_buy_button])
sell_controls_row = widgets.HBox([prev_sell_button, sell_dropdown, next_sell_button])
controls = widgets.VBox([buy_controls_row, sell_controls_row])

# Display widgets and plot
display(widgets.VBox([controls, stats_output, fig]))


# aggregated analysis

In [ ]:
# Calculate mean and std for each insertion across all samples
insertion_stats = []

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Check if we have insertion_number column, if not, we'll need to create it or use a different approach
if 'insertion_number' not in points_df.columns:
    print("Warning: 'insertion_number' column not found in points_df")
    print("Using sample-based analysis instead...")
    
    # Group by sample_id and create insertion numbers based on order
    points_df_with_insertion = points_df.copy()
    points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
    max_insertions = points_df_with_insertion['insertion_number'].max()
else:
    points_df_with_insertion = points_df
    max_insertions = points_df['insertion_number'].max()

# Show info about sample_id distribution
print("\nSample ID information:")
print(f"Total number of unique samples: {points_df_with_insertion['sample_id'].nunique()}")
print(f"Sample ID range: {points_df_with_insertion['sample_id'].min()} to {points_df_with_insertion['sample_id'].max()}")
print(f"Total data points: {len(points_df_with_insertion)}")

# Show sample distribution by insertion number
sample_counts = points_df_with_insertion.groupby('insertion_number')['sample_id'].nunique().reset_index()
sample_counts.columns = ['insertion_number', 'unique_samples']
print("\nSample distribution by insertion number:")
for _, row in sample_counts.iterrows():
    print(f"  Insertion {int(row['insertion_number'])}: {int(row['unique_samples'])} unique samples")

for insertion_num in range(1, max_insertions + 1):
    # Get data for this insertion number
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        x_values = insertion_data['x'].to_numpy()
        y_values = insertion_data['y'].to_numpy()
        sample_ids = insertion_data['sample_id'].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
        x_valid = x_values[valid_mask]
        y_valid = y_values[valid_mask]
        sample_ids_valid = sample_ids[valid_mask]
        
        if len(x_valid) > 0:
            x_mean = np.mean(x_valid)
            x_std = np.std(x_valid)
            y_mean = np.mean(y_valid)
            y_std = np.std(y_valid)
            
            insertion_stats.append({
                'insertion_number': insertion_num,
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_valid),
                'unique_samples': len(np.unique(sample_ids_valid)),
                'sample_ids': list(np.unique(sample_ids_valid))
            })

# Convert to DataFrame for easier handling
stats_df = pd.DataFrame(insertion_stats)

if len(stats_df) > 0:
    # Create the plotly figure
    fig = go.Figure()

    # Add scatter plot with error bars (only y-axis std)
    fig.add_trace(go.Scatter(
        x=stats_df['x_mean'],
        y=stats_df['y_mean'],
        error_y=dict(type='data', array=stats_df['y_std'], visible=True),
        mode='markers+text',
        text=[str(int(row['insertion_number'])) for _, row in stats_df.iterrows()],
        textposition='top right',
        textfont=dict(size=10),
        marker=dict(size=8, opacity=0.7),
        name=f'Mean impact by insertion (n={len(stats_df)})',
        hovertemplate='Insertion: %{text}<br>' +
                      'x_mean: %{x:.6f}<br>' +
                      'y_mean: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Add fitted line if we have enough points
    if len(stats_df) >= 2:
        x_mean_valid = stats_df['x_mean'].to_numpy()
        y_mean_valid = stats_df['y_mean'].to_numpy()
        
        # Fit line to mean values
        beta_insertion = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
        
        if np.isfinite(beta_insertion):
            x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
            y_fitted = alpha_global + beta_insertion * x_range
            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_fitted,
                mode='lines',
                line=dict(color='red', dash='dash', width=2),
                name=f'Fitted: β = {beta_insertion:.6f}',
                hovertemplate='Fitted line<br>' +
                              'x: %{x:.6f}<br>' +
                              'y: %{y:.6f}<br>' +
                              '<extra></extra>'
            ))

    # Add theoretical line
    beta_theory = 0.5
    x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
    y_theory = alpha_global + beta_theory * x_range_theory
    fig.add_trace(go.Scatter(
        x=x_range_theory,
        y=y_theory,
        mode='lines',
        line=dict(color='green', dash='dot', width=2),
        name=f'Theoretical: β = {beta_theory:.6f}',
        hovertemplate='Theoretical line<br>' +
                      'x: %{x:.6f}<br>' +
                      'y: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title='Market Impact: Mean by Insertion Number',
        xaxis_title='log(Q / V_exp) - Mean',
        yaxis_title='log(Impact) - Mean',
        width=1000,
        height=600,
        showlegend=True,
        hovermode='closest'
    )

    # Add grid
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

    # Display the plot
    fig.show()

    # Print summary statistics
    print(f"\nSummary Statistics:")
    print(f"Number of insertions analyzed: {len(stats_df)}")
    print(f"Alpha (fixed intercept): {alpha_global:.6f}")
    if len(stats_df) >= 2 and 'beta_insertion' in locals() and np.isfinite(beta_insertion):
        print(f"Fitted Beta (insertion means): {beta_insertion:.6f}")
    print(f"Theoretical Beta: {beta_theory:.6f}")
    print(f"Maximum insertion number: {max_insertions}")
    
    # Print detailed insertion statistics with sample info
    print(f"\nDetailed insertion statistics:")
    for _, row in stats_df.iterrows():
        print(f"  Insertion {int(row['insertion_number'])}: {int(row['n_points'])} points from {int(row['unique_samples'])} unique samples")
        print(f"    Sample IDs: {sorted(row['sample_ids'])}")
        print(f"    x_mean: {row['x_mean']:.6f} ± {row['x_std']:.6f}")
        print(f"    y_mean: {row['y_mean']:.6f} ± {row['y_std']:.6f}")
else:
    print("No valid insertion data found for analysis")


In [ ]:
# Calculate mean and std for each x-axis bin across all samples
import numpy as np

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Get all valid x and y values
x_values = points_df['x'].to_numpy()
y_values = points_df['y'].to_numpy()

# Filter out invalid points
valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
x_valid = x_values[valid_mask]
y_valid = y_values[valid_mask]

if len(x_valid) > 0:
    # Create bins for x-axis
    n_bins = 1000  # Number of bins
    x_min, x_max = x_valid.min(), x_valid.max()
    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Calculate statistics for each bin
    bin_stats = []
    
    for i in range(n_bins):
        # Find points in this bin
        bin_mask = (x_valid >= bin_edges[i]) & (x_valid < bin_edges[i + 1])
        if i == n_bins - 1:  # Include the last edge in the final bin
            bin_mask = (x_valid >= bin_edges[i]) & (x_valid <= bin_edges[i + 1])
        
        x_bin = x_valid[bin_mask]
        y_bin = y_valid[bin_mask]
        
        if len(x_bin) > 0:
            x_mean = np.mean(x_bin)
            x_std = np.std(x_bin)
            y_mean = np.mean(y_bin)
            y_std = np.std(y_bin)
            
            bin_stats.append({
                'bin_number': i + 1,
                'bin_center': bin_centers[i],
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_bin),
                'x_min': x_bin.min(),
                'x_max': x_bin.max()
            })

    # Convert to DataFrame for easier handling
    stats_df = pd.DataFrame(bin_stats)

    if len(stats_df) > 0:
        # Create the plotly figure
        fig = go.Figure()

        # Add scatter plot with error bars (only y-axis std)
        fig.add_trace(go.Scatter(
            x=stats_df['x_mean'],
            y=stats_df['y_mean'],
            error_y=dict(type='data', array=stats_df['y_std'], visible=True, color='blue'),
            mode='markers',
            marker=dict(size=5, color='black', opacity=1.0),
            name=f'Mean impact by x-bin (n={len(stats_df)})',
            hovertemplate='Bin: %{customdata}<br>' +
                          'x_mean: %{x:.6f}<br>' +
                          'y_mean: %{y:.6f}<br>' +
                          'n_points: %{customdata}<br>' +
                          '<extra></extra>',
            customdata=stats_df['bin_number']
        ))

        # Add fitted line if we have enough points
        if len(stats_df) >= 2:
            x_mean_valid = stats_df['x_mean'].to_numpy()
            y_mean_valid = stats_df['y_mean'].to_numpy()
            
            # Fit line to mean values
            beta_bins = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
            
            if np.isfinite(beta_bins):
                x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
                y_fitted = alpha_global + beta_bins * x_range
                fig.add_trace(go.Scatter(
                    x=x_range,
                    y=y_fitted,
                    mode='lines',
                    line=dict(color='red', dash='dash', width=2),
                    name=f'Fitted: β = {beta_bins:.6f}',
                    hovertemplate='Fitted line<br>' +
                                  'x: %{x:.6f}<br>' +
                                  'y: %{y:.6f}<br>' +
                                  '<extra></extra>'
                ))

        # Add theoretical line
        beta_theory = 0.5
        x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
        y_theory = alpha_global + beta_theory * x_range_theory
        fig.add_trace(go.Scatter(
            x=x_range_theory,
            y=y_theory,
            mode='lines',
            line=dict(color='green', dash='dot', width=2),
            name=f'Theoretical: β = {beta_theory:.6f}',
            hovertemplate='Theoretical line<br>' +
                          'x: %{x:.6f}<br>' +
                          'y: %{y:.6f}<br>' +
                          '<extra></extra>'
        ))

        # Update layout
        fig.update_layout(
            title='Market Impact: Mean by X-Axis Bins',
            xaxis_title='log(Q / V_exp) - Mean',
            yaxis_title='log(Impact) - Mean',
            width=1000,
            height=600,
            showlegend=True,
            hovermode='closest'
        )

        # Add grid
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

        # Display the plot
        fig.show()

        # Print summary statistics
        print(f"Number of x-bins analyzed: {len(stats_df)}")
        print(f"Number of bins: {n_bins}")
        print(f"Total points used: {len(x_valid)}")
        print(f"Alpha (fixed intercept): {alpha_global:.6f}")
        if len(stats_df) >= 2 and 'beta_bins' in locals() and np.isfinite(beta_bins):
            print(f"Fitted Beta (x-bin means): {beta_bins:.6f}")
        print(f"Theoretical Beta: {beta_theory:.6f}")
        print(f"X-axis range: [{x_min:.6f}, {x_max:.6f}]")
        
        # Print bin details
        print("\nBin details:")
        for _, row in stats_df.iterrows():
            print(f"Bin {int(row['bin_number'])}: x=[{row['x_min']:.6f}, {row['x_max']:.6f}], "
                  f"x_mean={row['x_mean']:.6f}, y_mean={row['y_mean']:.6f}, n={int(row['n_points'])}")
    else:
        print("No valid bin data found for analysis")
else:
    print("No valid data points found for analysis")


# Save data needed for mega plot

In [ ]:
import pickle
import os

# Define the experiment folder (replace with your actual experiment folder variable if needed)
experiment_folder = os.getcwd()  # or set to your experiment folder path

# Save all_betas_evo
with open(os.path.join(experiment_folder, "all_betas_evo.pkl"), "wb") as f:
    pickle.dump(all_betas_evo, f)

# Save insertion_data_dict
with open(os.path.join(experiment_folder, "insertion_data_dict.pkl"), "wb") as f:
    pickle.dump(insertion_data_dict, f)

In [ ]:
experiment_folder